# Retail Sales Dataset Exploratory Data Analysis

## Data Profiling

### Objective
The objective of this notebook is to understand the structure of the raw operational datasets before designing the ETL pipeline. This includes inspecting the schema, data types, dimensions and identifying potential primary and foreign keys.


In [2]:
import pandas as pd
import os

In [413]:
## Loading the Datasets

data_path = "../data"

datasets = {
    "customers": pd.read_csv(os.path.join(data_path, "olist_customers_dataset.csv")),
    "geolocation": pd.read_csv(os.path.join(data_path, "olist_geolocation_dataset.csv")),
    "order_items": pd.read_csv(os.path.join(data_path, "olist_order_items_dataset.csv")),
    "payments": pd.read_csv(os.path.join(data_path, "olist_order_payments_dataset.csv")),
    "reviews": pd.read_csv(os.path.join(data_path, "olist_order_reviews_dataset.csv")),
    "orders": pd.read_csv(os.path.join(data_path, "olist_orders_dataset.csv")),
    "products": pd.read_csv(os.path.join(data_path, "olist_products_dataset.csv")),
    "sellers": pd.read_csv(os.path.join(data_path, "olist_sellers_dataset.csv")),
    "category_translation": pd.read_csv(os.path.join(data_path, "product_category_name_translation.csv"))
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

customers: (99441, 5)
geolocation: (1000163, 5)
order_items: (112650, 7)
payments: (103886, 5)
reviews: (99224, 7)
orders: (99441, 8)
products: (32951, 9)
sellers: (3095, 4)
category_translation: (71, 2)


In [4]:
for name, df in datasets.items():

    print("=" * 60)
    print(f"Table Name : {name}")
    print("=" * 60)

    print(f"Rows    : {df.shape[0]}")
    print(f"Columns : {df.shape[1]}")

    print("\nData Types:")
    print(df.dtypes)

    print("\n")

Table Name : customers
Rows    : 99441
Columns : 5

Data Types:
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object


Table Name : geolocation
Rows    : 1000163
Columns : 5

Data Types:
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object


Table Name : order_items
Rows    : 112650
Columns : 7

Data Types:
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object


Table Name : payments
Rows    : 103886
Columns : 5

Data Types:
order_id                    str
payment_sequential        int64
payment_type                str
payment_ins

## Observations from above:

shipping_limit_date             str

review_creation_date             str

order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str

payments table has less number of rows than order items

orders has same no of rows as customers


In [5]:
customers = datasets["customers"]

print("Total Rows :", len(customers))
print("Unique Customer IDs :", customers["customer_id"].nunique())
print("Unique unique_cust_ids: ", customers["customer_unique_id"].nunique())
customers.groupby("customer_unique_id")["customer_id"].count().sort_values(ascending=False).head(10)

Total Rows : 99441
Unique Customer IDs : 99441
Unique unique_cust_ids:  96096


customer_unique_id
8d50f5eadf50201ccdcedfb9e2ac8455    17
3e43e6105506432c953e165fb2acf44c     9
1b6c7548a2a1f9037c1fd3ddfed95f33     7
6469f99c1f9dfae7733b25662e7f1782     7
ca77025e7201e3b30c44b472ff346268     7
47c1a3033b8b77b3ab6e109eb4d5fdf3     6
12f5d6e1cbf93dafd9dcc19095df0b3d     6
63cfc61cee11cbe306bff5857d00bfe4     6
dc813062e0fc23409cd255f7f53c7074     6
de34b16117594161a6a89c50b289d35a     6
Name: customer_id, dtype: int64

## Investigating Order-to-Product Relationship

One order may contain one or more products. This section verifies whether an order can have multiple entries in the `order_items` table.

In [6]:
order_items = datasets["order_items"]

order_items.groupby("order_id")["product_id"].count().sort_values(ascending=False).head(10)

order_id
8272b63d03f5f79c56e9e4120aec44ef    21
1b15974a0141d54e36626dca3fdc731a    20
ab14fdcfbe524636d65ee38360e22ce8    20
9ef13efd6949e4573a18964dd1bbe7f5    15
428a2f660dc84138d969ccd69a0ab6d5    15
9bdc4d4c71aa1de4606060929dee888c    14
73c8ab38f07dc94389065f7eba4f297a    14
37ee401157a3a0b28c9c6d0ed8c3b24b    13
af822dacd6f5cff7376413c03a388bb7    12
3a213fcdfe7d98be74ea0dc05a8b31ae    12
Name: product_id, dtype: int64

## Investigating Order-to-Payment Relationship

Each order may be associated with one or more payment records. This section verifies the relationship between the Orders and Payments tables.

In [7]:
payments = datasets["payments"]

payments.groupby("order_id")["payment_sequential"].count().sort_values(ascending=False).head(10)

order_id
fa65dad1b0e818e3ccc5cb0e39231352    29
ccf804e764ed5650cd8759557269dc13    26
285c2e15bebd4ac83635ccc563dc71f4    22
895ab968e7bb0d5659d16cd74cd1650c    21
fedcd9f7ccdc8cba3a18defedd1a5547    19
ee9ca989fc93ba09a6eddc250ce01742    19
4bfcba9e084f46c8e3cb49b0fa6e6159    15
21577126c19bf11a0b91592e5844ba78    15
4689b1816de42507a7d63a4617383c59    14
3c58bffb70dcf45f12bdf66a3c215905    14
Name: payment_sequential, dtype: int64

In [8]:
payments.loc[payments["order_id"] == "fa65dad1b0e818e3ccc5cb0e39231352"].sort_values(by="payment_sequential")

,order_id,payment_sequential,payment_type,payment_installments,payment_value
14321,fa65dad1b0e818e3ccc5cb0e39231352,1,voucher,1,3.71
23074,fa65dad1b0e818e3ccc5cb0e39231352,2,voucher,1,8.51
65641,fa65dad1b0e818e3ccc5cb0e39231352,3,voucher,1,2.95
9985,fa65dad1b0e818e3ccc5cb0e39231352,4,voucher,1,29.16
28330,fa65dad1b0e818e3ccc5cb0e39231352,5,voucher,1,0.66
29648,fa65dad1b0e818e3ccc5cb0e39231352,6,voucher,1,5.02
82593,fa65dad1b0e818e3ccc5cb0e39231352,7,voucher,1,0.32
68853,fa65dad1b0e818e3ccc5cb0e39231352,8,voucher,1,26.02
17274,fa65dad1b0e818e3ccc5cb0e39231352,9,voucher,1,1.08
19565,fa65dad1b0e818e3ccc5cb0e39231352,10,voucher,1,12.86


#### Observation

The analysis shows that a single order can contain multiple payment records. Inspecting an order with the highest number of payment records revealed that the `payment_sequential` values increase consecutively from 1 to 29.

#### Conclusion

Based on the observed data, `payment_sequential` appears to represent the sequence number of payment records within an order. This indicates that the relationship between the `orders` and `payments` tables is **one-to-many**, where a single order may be associated with multiple payment entries.

In [9]:
payments.groupby("order_id").size().value_counts().sort_index()

1     96479
2      2382
3       301
4       108
5        52
6        36
7        28
8        11
9         9
10        5
11        8
12        8
13        3
14        2
15        2
19        2
21        1
22        1
26        1
29        1
Name: count, dtype: int64

In [10]:
orders = datasets["orders"]
print(payments[~payments["order_id"].isin(orders["order_id"])])

Empty DataFrame
Columns: [order_id, payment_sequential, payment_type, payment_installments, payment_value]
Index: []


In [11]:
order_items = datasets["order_items"]
order_items.head(10)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
5,00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23 03:55:27,21.90,12.69
6,00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14 12:10:31,19.90,11.85
7,000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10 12:30:45,810.00,70.75
8,0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26 18:31:29,145.95,11.65
9,0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06 14:10:56,53.99,11.40


In [12]:
order_items[order_items["order_id"] == "00143d0f86d6fbd9f9b38ab440ac16f5"]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
32,00143d0f86d6fbd9f9b38ab440ac16f5,1,e95ee6822b66ac6058e2e4aff656071a,a17f621c590ea0fab3d5d883e1630ec6,2017-10-20 16:07:52,21.33,15.1
33,00143d0f86d6fbd9f9b38ab440ac16f5,2,e95ee6822b66ac6058e2e4aff656071a,a17f621c590ea0fab3d5d883e1630ec6,2017-10-20 16:07:52,21.33,15.1
34,00143d0f86d6fbd9f9b38ab440ac16f5,3,e95ee6822b66ac6058e2e4aff656071a,a17f621c590ea0fab3d5d883e1630ec6,2017-10-20 16:07:52,21.33,15.1


In [13]:
order_items[order_items["order_id"] == "001ab0a7578dd66cd4b0a71f5b6e1e41"]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
42,001ab0a7578dd66cd4b0a71f5b6e1e41,1,0b0172eb0fd18479d29c3bc122c058c2,5656537e588803a555b8eb41f07a944b,2018-01-04 02:33:42,24.89,17.63
43,001ab0a7578dd66cd4b0a71f5b6e1e41,2,0b0172eb0fd18479d29c3bc122c058c2,5656537e588803a555b8eb41f07a944b,2018-01-04 02:33:42,24.89,17.63
44,001ab0a7578dd66cd4b0a71f5b6e1e41,3,0b0172eb0fd18479d29c3bc122c058c2,5656537e588803a555b8eb41f07a944b,2018-01-04 02:33:42,24.89,17.63


In [14]:
order_items[order_items["order_id"] == "001d8f0e34a38c37f7dba2a37d4eba8b"]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
48,001d8f0e34a38c37f7dba2a37d4eba8b,1,e67307ff0f15ade43fcb6e670be7a74c,f4aba7c0bca51484c30ab7bdc34bcdd1,2017-05-18 17:35:11,18.99,7.78
49,001d8f0e34a38c37f7dba2a37d4eba8b,2,e67307ff0f15ade43fcb6e670be7a74c,f4aba7c0bca51484c30ab7bdc34bcdd1,2017-05-18 17:35:11,18.99,7.78


In [15]:
# Is order_id unique?
order_items["order_id"].nunique(), len(order_items)

(98666, 112650)

In [16]:
# Is order_item_id unique?
order_items["order_item_id"].nunique(), len(order_items)

(21, 112650)

In [17]:
# Is the combination unique?
order_items.duplicated(subset=["order_id", "order_item_id"]).sum()

np.int64(0)

## Order Items Table - Primary Key Validation

Three possible candidates are evaluated to identify the primary key of the `order_items` table.

- `order_id` is **not unique**, as a single order may contain multiple purchased items.
- `order_item_id` is **not unique**, as numbering restarts from 1 for each new order.
- The combination of (`order_id`, `order_item_id`) contains **no duplicate records**, indicating that together they uniquely identify each purchased item.

**Conclusion:** The `order_items` table uses a **composite primary key** consisting of `order_id` and `order_item_id`.

In [18]:
items_per_order = order_items.groupby("order_id")["order_item_id"].count().reset_index()
print(items_per_order)
multiple_item_orders = len(items_per_order.loc[items_per_order["order_item_id"] > 1])
print(multiple_item_orders)

multiple_item_orders_pct = (multiple_item_orders / order_items["order_id"].nunique()) * 100
print(multiple_item_orders_pct,"%")

                               order_id  order_item_id
0      00010242fe8c5a6d1ba2dd792cb16214              1
1      00018f77f2f0320c557190d7a144bdd3              1
2      000229ec398224ef6ca0657da4fc703e              1
3      00024acbcdf0a6daa1e931b038114c75              1
4      00042b26cf59d7ce69dfabb4e55b4fd9              1
...                                 ...            ...
98661  fffc94f6ce00a00581880bf54a75a037              1
98662  fffcd46ef2263f404302a634eb57f7eb              1
98663  fffce4705a9662cd70adb13d4a31832d              1
98664  fffe18544ffabc95dfada21779c9644f              1
98665  fffe41c64501cc87c801fd61db3f6244              1

[98666 rows x 2 columns]
9803
9.93554010500071 %


In [19]:
orders.head()
#orders.info()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [20]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
print(orders["order_purchase_timestamp"].count())

99441


In [21]:
date_columns = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders[column] = pd.to_datetime(orders[column])

'''
orders[[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]].dtypes
'''

'\norders[[\n        "order_purchase_timestamp",\n        "order_approved_at",\n        "order_delivered_carrier_date",\n        "order_delivered_customer_date",\n        "order_estimated_delivery_date"\n    ]].dtypes\n'

In [22]:
orders[
    ["order_purchase_timestamp",
    "order_delivered_customer_date"]
].isna().sum()

order_purchase_timestamp            0
order_delivered_customer_date    2965
dtype: int64

In [23]:
orders.loc[orders["order_delivered_customer_date"].isna(),"order_status"].value_counts()

orders.loc[
    orders["order_delivered_customer_date"].isna() & orders["order_status"].isin(["delivered"]),
    [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
]


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19


### Missing Customer Delivery Dates

The `order_delivered_customer_date` column contains 2,965 missing values.

Most missing values are associated with orders that had not reached the customer-delivery stage, such as `shipped`, `canceled`, `unavailable`, `invoiced`, and `processing` orders. Missing delivery dates for these statuses are therefore expected.

A small anomaly was identified: 8 orders have `order_status = delivered` but no `order_delivered_customer_date`. These records will not be assigned an inferred delivery date and will be excluded from analyses that require an actual customer delivery timestamp.

In [24]:
orders["delivery_time_days"] = (orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]).dt.total_seconds() / (24*60*60)
print(orders["delivery_time_days"].head(25))
orders["delivery_time_days"].describe()

0      8.436574
1     13.782037
2      9.394213
3     13.208750
4      2.873877
5     16.542245
6           NaN
7      9.989826
8      9.818762
9     18.221852
10    12.650937
11     5.754595
12    12.083021
13     4.922396
14    11.390255
15    13.855093
16    17.276574
17    12.954919
18     6.766887
19    13.269144
20    21.327963
21     4.386019
22     4.816794
23    18.400799
24     4.277627
Name: delivery_time_days, dtype: float64


count    96476.000000
mean        12.558702
std          9.546530
min          0.533414
25%          6.766403
50%         10.217755
75%         15.720327
max        209.628611
Name: delivery_time_days, dtype: float64

In [25]:
orders.loc[orders["delivery_time_days"] == orders["delivery_time_days"].max(),
    [

            "order_id",
            "customer_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date"

    ]

]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
19590,ca07593549f1816d26a572e06dc1eab6,75683a92331068e2d281b11a7866ba44,delivered,2017-02-21 23:31:27,2017-02-23 02:35:15,2017-03-08 13:47:46,2017-09-19 14:36:39,2017-03-22


In [26]:
orders["delivery_time_days"].value_counts(
    bins=[0, 5, 10, 15, 20, 30, 60, 100, 250]
).sort_index()

(-0.001, 5.0]     13430
(5.0, 10.0]       33052
(10.0, 15.0]      23614
(15.0, 20.0]      12154
(20.0, 30.0]       9673
(30.0, 60.0]       4247
(60.0, 100.0]       242
(100.0, 250.0]       64
Name: count, dtype: int64

In [27]:
orders["delivery_delay_days"] = (orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]).dt.total_seconds() / (24*60*60)
orders["delivery_delay_days"].describe()

orders.loc[orders["delivery_delay_days"] < 0, ["delivery_delay_type"]] = "Early"
orders.loc[orders["delivery_delay_days"] == 0, ["delivery_delay_type"]] = "On time"
orders.loc[orders["delivery_delay_days"] > 0, ["delivery_delay_type"]] = "Late"

orders["delivery_delay_type"].value_counts()

orders.loc[
    orders["order_delivered_customer_date"].dt.date
    == orders["order_estimated_delivery_date"].dt.date,
    [
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_delay_days"
    ]
].head()

orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"].dt.normalize() - orders["order_estimated_delivery_date"].dt.normalize()
).dt.days

orders["delivery_delay_days"].describe()

orders.loc[orders["delivery_delay_days"] < 0, "delivery_delay_type"] = "Early"
orders.loc[orders["delivery_delay_days"] == 0, "delivery_delay_type"] = "On time"
orders.loc[orders["delivery_delay_days"] > 0, "delivery_delay_type"] = "Late"

orders["delivery_delay_type"].value_counts()

delivery_delay_type
Early      88649
Late        6535
On time     1292
Name: count, dtype: int64

### Delivery Performance
Delivery performance was classified by comparing the actual customer delivery date with the estimated delivery date at calendar-day level.
- Early: 88,649 orders (91.89%)
- On time: 1,292 orders (1.34%)
- Late: 6,535 orders (6.77%)

The majority of delivered orders arrived before the estimated delivery date.

In [28]:
orders["order_status"].value_counts()
orders["order_status"].value_counts(normalize=True)

order_status
delivered      0.970203
shipped        0.011132
canceled       0.006285
unavailable    0.006124
invoiced       0.003158
processing     0.003027
created        0.000050
approved       0.000020
Name: proportion, dtype: float64

In [29]:
payments.head()
payments.info()

payments["order_id"].nunique(), len(payments)

payment_record_count = (
    payments
    .groupby("order_id")["payment_sequential"]
    .count()
    .value_counts()
    .sort_index()
    .rename_axis("payment_records_per_order")
    .reset_index(name="number_of_orders")
)
print(payment_record_count)

payments.head()


<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 4.0 MB
    payment_records_per_order  number_of_orders
0                           1             96479
1                           2              2382
2                           3               301
3                           4               108
4                           5                52
5                           6                36
6                           7                28
7                           8                11
8                           9                 9

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


### Logic for rename_axis and reset_index name parameter
GROUP BY
↓
How many payment rows does each order have?

VALUE_COUNTS
↓
How many orders have each particular number of payment rows?

SORT_INDEX
↓
Put 1, 2, 3, 4... in numerical order

RENAME_AXIS
↓
Give that index a meaningful name

RESET_INDEX
↓
Turn the index into a normal DataFrame column

NAME=
↓
Give the resulting count column a meaningful name

In [30]:
payments["payment_type"].value_counts()

payment_types_per_order = payments.groupby("order_id")["payment_type"].nunique()
#print(payment_types_per_order.value_counts().reset_index())

orders_per_payment_types = payments.groupby("payment_type")["order_id"].nunique()
print(orders_per_payment_types.reset_index(name="number_of_orders").sort_values("number_of_orders", ascending=False))

  payment_type  number_of_orders
1  credit_card             76505
0       boleto             19784
4      voucher              3866
2   debit_card              1528
3  not_defined                 3


In [31]:
payments["payment_installments"].value_counts().sort_index()

payments.loc[payments["payment_installments"] == 0]

payments.loc[(payments["order_id"] == "744bade1fcf9ff3f31d860ace076d422") | (payments["order_id"] == "1a57108394169c0b47d8f876acc9ba2d")]

#payments.groupby("payment_sequential")["payment_installments"].apply(lambda x: (x == 0).sum())

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [32]:
payments.loc[
    payments["order_id"].isin([
        "744bade1fcf9ff3f31d860ace076d422",
        "1a57108394169c0b47d8f876acc9ba2d"]
    )
].sort_values(["order_id", "payment_sequential"])

,order_id,payment_sequential,payment_type,payment_installments,payment_value
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69


Two anomalous records with payment sequential being 2 and payment installments being 0 were identified, but their cause cannot be established from the available fields. They should be retained rather than arbitrarily modified.

In [33]:
payments["payment_value"].describe()

payments.loc[payments["payment_value"] == 0]

payments.loc[payments["order_id"].isin(payments.loc[payments["payment_value"] == 0,"order_id"])].sort_values(["order_id","payment_sequential"])

,order_id,payment_sequential,payment_type,payment_installments,payment_value
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00
33781,45ed6e85398a87c253db47c2d9f48216,1,voucher,1,21.13
11755,45ed6e85398a87c253db47c2d9f48216,2,voucher,1,50.01
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00
40546,6ccb433e00daae1283ccc956189c82ae,1,credit_card,5,84.67
93478,6ccb433e00daae1283ccc956189c82ae,2,voucher,1,14.65
92318,6ccb433e00daae1283ccc956189c82ae,3,voucher,1,22.72
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00
20963,8bcbe01d44d147f901cd3192671144db,1,credit_card,1,36.21


Nine zero-value payment records were identified. All were associated with either voucher or not_defined payment types. Their sequence positions varied, and the available fields do not provide enough evidence to determine why these records have zero value. They are sufficiently rare that they are unlikely to materially affect aggregate payment-value analysis.

In [34]:
payment_value_per_order = payments.groupby("order_id")["payment_value"].sum()
#print(payment_value_per_order.sort_values(ascending=False))
payment_value_per_order.describe()

count    99440.000000
mean       160.990267
std        221.951257
min          0.000000
25%         62.010000
50%        105.290000
75%        176.970000
max      13664.080000
Name: payment_value, dtype: float64

In [35]:
customers.head()
customers.info()

len(customers), customers["customer_id"].nunique(), customers["customer_unique_id"].nunique()


<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB


(99441, 99441, 96096)

In [36]:
no_of_customerids_per_unique_id = customers.groupby("customer_unique_id")["customer_id"].count()
no_of_customerids_per_unique_id.value_counts()

repeated_customer = no_of_customerids_per_unique_id[no_of_customerids_per_unique_id > 1].index[0]
repeated_customer

customers.loc[customers["customer_unique_id"] == repeated_customer]

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
19299,1afe8a9c67eec3516c09a8bdcc539090,00172711b30d52eea8b313a7f2cced02,45200,jequie,BA
35608,24b0e2bd287e47d54d193e7bbb51103f,00172711b30d52eea8b313a7f2cced02,45200,jequie,BA


In [37]:
repeated_customers = no_of_customerids_per_unique_id[no_of_customerids_per_unique_id > 1].index
location_check = (
    customers[customers["customer_unique_id"].isin(repeated_customers)]
    .groupby("customer_unique_id")
    [["customer_zip_code_prefix", "customer_city", "customer_state"]]
    .nunique()
)

location_check[
    (location_check["customer_zip_code_prefix"] > 1) |
    (location_check["customer_city"] > 1) |
    (location_check["customer_state"] > 1)
]

location_check[
    (location_check["customer_zip_code_prefix"] > 1) |
    (location_check["customer_city"] > 1) |
    (location_check["customer_state"] > 1)
].value_counts().sort_index()

customer_zip_code_prefix  customer_city  customer_state
1                         2              1                   2
2                         1              1                 129
                          2              1                  80
                                         2                  37
3                         1              1                   1
                          2              1                   1
                                         2                   1
                          3              3                   1
Name: count, dtype: int64

In [38]:
orders.loc[
    orders["customer_id"].isin(
        customers.loc[
            customers["customer_unique_id"] == repeated_customer,
            "customer_id"
        ]
    )
]

location_check["customer_state"].value_counts()
#location_check["customer_city"].value_counts()
#location_check["customer_zip_code_prefix"].value_counts()

location_check[location_check["customer_state"] > 1]

,customer_zip_code_prefix,customer_city,customer_state
customer_unique_id,,,
1f90117a847636892e3c5bf569f2ac68,2,2,2
2410195f6521688005612363835a2671,2,2,2
2c45ab66a3dae52960147e76a35740ff,2,2,2
2c6a91479a7dc00d8c9d650d8dee88ca,2,2,2
408aee96c75632a92e5079eee61da399,2,2,2
5192c897072033288df55bd01b0e5737,2,2,2
5275b2f97b9c995d3d05a58610c0bb67,2,2,2
547d0504ca415eb4864fa3030f73d3f3,2,2,2
5cbfdb85ec130898108b32c50d619c39,2,2,2


In [39]:
customers[customers["customer_unique_id"] == "1f90117a847636892e3c5bf569f2ac68"]

orders.loc[orders["customer_id"].isin(["b0a9bd08e4116c3fdd62aebc51031f19","36d63113a38882f3f464f8eaa96c4b1c"])].sort_values(by="order_purchase_timestamp")

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_time_days,delivery_delay_days,delivery_delay_type
42071,8d0584fef2164e00c8eb23abbdd87ef7,b0a9bd08e4116c3fdd62aebc51031f19,delivered,2017-08-13 20:03:35,2017-08-13 20:15:19,2017-08-14 20:05:01,2017-08-15 18:03:20,2017-08-25,1.916493,-10.0,Early
87783,00c42b1633c1b2fa4e937bf312466561,36d63113a38882f3f464f8eaa96c4b1c,delivered,2018-01-13 13:52:54,2018-01-13 14:09:30,2018-01-15 19:32:48,2018-02-08 21:12:51,2018-02-15,26.305521,-7.0,Early


Key takeaway: customer_unique_id is the best field for identifying recurring customers across records/orders. customer_id is unique to each customer record. Most recurring customers have consistent location information, but a small number appear with multiple locations/states, so location should not be used as proof of customer identity.

In [40]:
unique_customers_by_state = customers.groupby("customer_state")["customer_unique_id"].nunique().sort_values(ascending=False)

unique_customers_by_state_pct = unique_customers_by_state / unique_customers_by_state.sum() * 100
unique_customers_by_state_pct

customer_state
SP    41.921861
RJ    12.881751
MG    11.711534
RS     5.489099
PR     5.078223
SC     3.676042
BA     3.408713
DF     2.158401
ES     2.042939
GO     2.030457
PE     1.673671
CE     1.365773
PA     0.987143
MT     0.911209
MA     0.755180
MS     0.721894
PB     0.539860
PI     0.501373
RN     0.493052
AL     0.417117
SE     0.355746
TO     0.283973
RO     0.249646
AM     0.148748
AC     0.080095
AP     0.069693
RR     0.046809
Name: customer_unique_id, dtype: float64

In [41]:
unique_customers_by_city = customers.groupby("customer_city")["customer_unique_id"].nunique().sort_values(ascending=False)

unique_customers_by_city.head(10)

unique_customers_by_city.sum(), customers["customer_unique_id"].nunique()

unique_customers_by_city_pct = unique_customers_by_city / customers["customer_unique_id"].nunique() * 100
unique_customers_by_city_pct.head(10)

customer_city
sao paulo                15.592741
rio de janeiro            6.888944
belo horizonte            2.780553
brasilia                  2.153055
curitiba                  1.524517
campinas                  1.454795
porto alegre              1.379870
salvador                  1.258117
guarulhos                 1.199842
sao bernardo do campo     0.944888
Name: customer_unique_id, dtype: float64

### Why use `customer_unique_id.nunique()` instead of `unique_customers_by_city.sum()`?

We want the percentage of the total **unique customer base** represented by the cities.

`unique_customers_by_city` contains the number of unique customers in each city. However, some `customer_unique_id`s appear in more than one city. Therefore, when we sum the city-level counts using:

`unique_customers_by_city.sum()`

those customers are counted once for each city they appear in.

For example, if one customer appears in 2 cities, they are counted as a unique customer in both those cities, where in reality there are one unique customer only. So they count for 1 extra unique record despite not being one.

This is why:

- `unique_customers_by_city.sum()` = 96,219
- `customers["customer_unique_id"].nunique()` = 96,096

The difference of 123 represents these additional city assignments. {See the output of - location_check[
    (location_check["customer_zip_code_prefix"] > 1) |
    (location_check["customer_city"] > 1) |
    (location_check["customer_state"] > 1)
].value_counts().sort_index() - above}

Therefore, for the total customer base, we must use:
`customers["customer_unique_id"].nunique()`
because it counts each underlying customer exactly once, regardless of how many cities they appear in.

**Conclusion:**
Use `unique_customers_by_city.sum()` when we want the total of the city-level counts. Use `customers["customer_unique_id"].nunique()` when we want the actual total number of unique customers.

In [42]:
top_10_city_pct = (
    unique_customers_by_city.head(10).sum()
    / customers["customer_unique_id"].nunique()
) * 100

top_10_city_pct

np.float64(35.17732267732268)

In [43]:
order_items.head(10)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
5,00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23 03:55:27,21.90,12.69
6,00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14 12:10:31,19.90,11.85
7,000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10 12:30:45,810.00,70.75
8,0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26 18:31:29,145.95,11.65
9,0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06 14:10:56,53.99,11.40


In [44]:
order_items.info()

order_items["order_item_id"].value_counts().sort_index()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB


order_item_id
1     98666
2      9803
3      2287
4       965
5       460
6       256
7        58
8        36
9        28
10       25
11       17
12       13
13        8
14        7
15        5
16        3
17        3
18        3
19        3
20        3
21        1
Name: count, dtype: int64

In [45]:
order_items.groupby("order_id")["order_item_id"].count().describe()

items_per_order = order_items.groupby("order_id")["order_item_id"].count()
items_per_order.value_counts().sort_index()

order_item_id
1     88863
2      7516
3      1322
4       505
5       204
6       198
7        22
8         8
9         3
10        8
11        4
12        5
13        1
14        2
15        2
20        2
21        1
Name: count, dtype: int64

In [46]:
order_items[["price","freight_value"]].describe()

order_value = (
    order_items.groupby("order_id")
    .agg(
        total_product_value=("price", "sum"),
        total_freight_value=("freight_value", "sum"),
        item_count=("order_item_id", "count")
    )
)

order_value.describe()
order_value

,total_product_value,total_freight_value,item_count
order_id,,,
00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,1
00018f77f2f0320c557190d7a144bdd3,239.90,19.93,1
000229ec398224ef6ca0657da4fc703e,199.00,17.87,1
00024acbcdf0a6daa1e931b038114c75,12.99,12.79,1
00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,1
...,...,...,...
fffc94f6ce00a00581880bf54a75a037,299.99,43.41,1
fffcd46ef2263f404302a634eb57f7eb,350.00,36.53,1
fffce4705a9662cd70adb13d4a31832d,99.90,16.95,1


Why do we care about order-level totals? Because businesses generally care about things like:
How much does a typical order generate?

rather than:
How much does a typical individual item cost?

In [47]:
order_value["freight_pct_of_product_value"] = order_value["total_freight_value"] / order_value["total_product_value"] * 100
order_value
order_value["freight_pct_of_product_value"].describe()

count    98666.000000
mean        30.838869
std         31.476219
min          0.000000
25%         13.186441
50%         22.437396
75%         38.019087
max       2144.705882
Name: freight_pct_of_product_value, dtype: float64

In [48]:
order_value.nlargest(10,"freight_pct_of_product_value")[["total_product_value","total_freight_value","item_count","freight_pct_of_product_value"]]

,total_product_value,total_freight_value,item_count,freight_pct_of_product_value
order_id,,,,
3ee6513ae7ea23bdfab5b9ab60bffcb5,0.85,18.23,1,2144.705882
6e864b3f0ec71031117ad4cf46b7f2a1,0.85,18.23,1,2144.705882
fb265b2dc558a56445dfc48f8224e201,9.90,121.22,1,1224.444444
d642656598ae928a250620315d19e87e,4.99,37.04,1,742.284569
7c92284adbec8033d160303d83065dbf,9.18,54.69,3,595.751634
b2d30dfaf7114de8ab3f737d062032fe,5.95,34.15,1,573.949580
21fb84496c034392ca2aafe7f7919a3f,6.99,37.90,1,542.203147
a16607f4220f0e60ab4c71b915dd380c,14.00,74.08,2,529.142857
95d6357ffe41aa6d2998852a710c70a0,17.50,91.15,5,520.857143


In [49]:
order_items.duplicated().sum()
order_items.duplicated(subset=["order_id", "order_item_id"]).sum()

np.int64(0)

A reminder is - whenever we see two columns used in .duplicated() in subset=[A,B], it essentially means that we are not checking columns A and B separately to check whether there are having duplicate records in them, but rather we are checking the possibility of exact similar records in those two columns.

In [50]:
top_10_extreme_freight_orders = order_value.nlargest(10, "freight_pct_of_product_value").reset_index()
top_10_extreme_freight_orders

,order_id,total_product_value,total_freight_value,item_count,freight_pct_of_product_value
0,3ee6513ae7ea23bdfab5b9ab60bffcb5,0.85,18.23,1,2144.705882
1,6e864b3f0ec71031117ad4cf46b7f2a1,0.85,18.23,1,2144.705882
2,fb265b2dc558a56445dfc48f8224e201,9.90,121.22,1,1224.444444
3,d642656598ae928a250620315d19e87e,4.99,37.04,1,742.284569
4,7c92284adbec8033d160303d83065dbf,9.18,54.69,3,595.751634
5,b2d30dfaf7114de8ab3f737d062032fe,5.95,34.15,1,573.949580
6,21fb84496c034392ca2aafe7f7919a3f,6.99,37.90,1,542.203147
7,a16607f4220f0e60ab4c71b915dd380c,14.00,74.08,2,529.142857
8,95d6357ffe41aa6d2998852a710c70a0,17.50,91.15,5,520.857143
9,8272b63d03f5f79c56e9e4120aec44ef,31.80,164.37,21,516.886792


In [51]:
order_items.loc[order_items["order_id"] == "8272b63d03f5f79c56e9e4120aec44ef"]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
57297,8272b63d03f5f79c56e9e4120aec44ef,1,270516a3f41dc035aa87d220228f844c,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57298,8272b63d03f5f79c56e9e4120aec44ef,2,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57299,8272b63d03f5f79c56e9e4120aec44ef,3,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57300,8272b63d03f5f79c56e9e4120aec44ef,4,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57301,8272b63d03f5f79c56e9e4120aec44ef,5,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57302,8272b63d03f5f79c56e9e4120aec44ef,6,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57303,8272b63d03f5f79c56e9e4120aec44ef,7,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57304,8272b63d03f5f79c56e9e4120aec44ef,8,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57305,8272b63d03f5f79c56e9e4120aec44ef,9,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57306,8272b63d03f5f79c56e9e4120aec44ef,10,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89


In [52]:
order_items.groupby(["order_id", "product_id"]).size().sort_values(ascending=False).head(20)

order_id                          product_id                      
ab14fdcfbe524636d65ee38360e22ce8  9571759451b1d780ee7c15012ea109d4    20
1b15974a0141d54e36626dca3fdc731a  ee3d532c8a438679776d222e997606b3    20
428a2f660dc84138d969ccd69a0ab6d5  89b190a046022486c635022524a974a8    15
9ef13efd6949e4573a18964dd1bbe7f5  37eb69aca8718e843d897aa7b82f462d    15
73c8ab38f07dc94389065f7eba4f297a  422879e10f46682990de24d770e7f83d    14
9bdc4d4c71aa1de4606060929dee888c  44a5d24dd383324a421569ca697b13c2    14
37ee401157a3a0b28c9c6d0ed8c3b24b  d34c07a2d817ac73f4caf8c574215fed    13
2c2a19b5703863c908512d135aa6accc  03e1c946c0ddfc58724ff262aef08dff    12
3a213fcdfe7d98be74ea0dc05a8b31ae  a62e25e09e05e6faf31d90c6ec1aa3d1    12
6c355e2913545fa6f72c40cbca57729e  32e18e89237933ebdaaebd78a27e7fa1    11
8272b63d03f5f79c56e9e4120aec44ef  05b515fdc76e888aada3c6d66c201dff    10
f80549a97eb203e1566e026ab66f045b  d5991653e037ccb7af6ed7d94246b249    10
a483ffe0ce133740ab12ebcba8a3ccf9  36f60d45225e60c7da4558b

In [53]:
order_items[["order_id", "product_id"]].value_counts().sort_values(ascending=False).head(21)

order_id                          product_id                      
1b15974a0141d54e36626dca3fdc731a  ee3d532c8a438679776d222e997606b3    20
ab14fdcfbe524636d65ee38360e22ce8  9571759451b1d780ee7c15012ea109d4    20
428a2f660dc84138d969ccd69a0ab6d5  89b190a046022486c635022524a974a8    15
9ef13efd6949e4573a18964dd1bbe7f5  37eb69aca8718e843d897aa7b82f462d    15
73c8ab38f07dc94389065f7eba4f297a  422879e10f46682990de24d770e7f83d    14
9bdc4d4c71aa1de4606060929dee888c  44a5d24dd383324a421569ca697b13c2    14
37ee401157a3a0b28c9c6d0ed8c3b24b  d34c07a2d817ac73f4caf8c574215fed    13
2c2a19b5703863c908512d135aa6accc  03e1c946c0ddfc58724ff262aef08dff    12
3a213fcdfe7d98be74ea0dc05a8b31ae  a62e25e09e05e6faf31d90c6ec1aa3d1    12
6c355e2913545fa6f72c40cbca57729e  32e18e89237933ebdaaebd78a27e7fa1    11
8272b63d03f5f79c56e9e4120aec44ef  270516a3f41dc035aa87d220228f844c    10
                                  05b515fdc76e888aada3c6d66c201dff    10
a483ffe0ce133740ab12ebcba8a3ccf9  36f60d45225e60c7da4558b

The above analysis of .size() and .value_counts() confirms that there are orders within this dataset that are having multiple items under one order id. This was done to verify that there are no duplicate records present within the dataset, as there was an order with 21 order item ids that raised suspicion about duplicacy.

In [54]:
print(order_items["product_id"].nunique())

order_items.groupby("product_id").size().sort_values(ascending=False).head(10)

32951


product_id
aca2eb7d00ea1a7b8ebd4e68314663af    527
99a4788cb24856965c36a24e339b6058    488
422879e10f46682990de24d770e7f83d    484
389d119b48cf3043d311335e499d9c6b    392
368c6c730842d78016ad823897a372db    388
53759a2ecddad2bb87a079a1f1519f73    373
d1c427060a0f73f6b889a5c7c61f2ac4    343
53b36df67ebb7c41585e8d54d6772e08    323
154e7e31ebfa092203795c972e5804a6    281
3dd2a17168ec895c781a9191c1e95ad7    274
dtype: int64

In [55]:
product_sales = (
    order_items.groupby("product_id")
    .agg(
        units_sold=("order_item_id", "count"),
        orders_with_product=("order_id", "nunique")
    )
    .sort_values("units_sold", ascending=False)
)

product_sales.head(15)

,units_sold,orders_with_product
product_id,,
aca2eb7d00ea1a7b8ebd4e68314663af,527,431
99a4788cb24856965c36a24e339b6058,488,467
422879e10f46682990de24d770e7f83d,484,352
389d119b48cf3043d311335e499d9c6b,392,311
368c6c730842d78016ad823897a372db,388,291
53759a2ecddad2bb87a079a1f1519f73,373,287
d1c427060a0f73f6b889a5c7c61f2ac4,343,323
53b36df67ebb7c41585e8d54d6772e08,323,306
154e7e31ebfa092203795c972e5804a6,281,269


In [56]:
product_sales["avg_units_per_order"] = product_sales["units_sold"] / product_sales["orders_with_product"]
product_sales.sort_values(by="avg_units_per_order",ascending = False).head(10)

,units_sold,orders_with_product,avg_units_per_order
product_id,,,
9571759451b1d780ee7c15012ea109d4,20,1,20.0
37eb69aca8718e843d897aa7b82f462d,15,1,15.0
05b515fdc76e888aada3c6d66c201dff,10,1,10.0
270516a3f41dc035aa87d220228f844c,10,1,10.0
89b190a046022486c635022524a974a8,20,2,10.0
5769ef0a239114ac3a854af00df129e4,8,1,8.0
810cfa5dd36b001cfc186499381f72ab,7,1,7.0
710b7c26b7a742f497bba45fab91a25f,6,1,6.0
ac1ad58efc1ebf66bfadc09f29bdedc0,6,1,6.0


In [57]:
product_sales["total_sales_value"] = order_items.groupby("product_id")["price"].sum()

product_sales.sort_values(by="total_sales_value",ascending = False).head(10)

,units_sold,orders_with_product,avg_units_per_order,total_sales_value
product_id,,,,
bb50f2e236e5eea0100680137654686c,195,187,1.042781,63885.00
6cdd53843498f92890544667809f1595,156,151,1.033113,54730.20
d6160fb7873f184099d9bc95e30376af,35,35,1.000000,48899.34
d1c427060a0f73f6b889a5c7c61f2ac4,343,323,1.061920,47214.51
99a4788cb24856965c36a24e339b6058,488,467,1.044968,43025.56
3dd2a17168ec895c781a9191c1e95ad7,274,255,1.074510,41082.60
25c38557cf793876c5abdd5931f922db,38,38,1.000000,38907.32
5f504b3a1c75b73d6151be81eb05bdc9,63,63,1.000000,37733.90
53b36df67ebb7c41585e8d54d6772e08,323,306,1.055556,37683.42


In [58]:
product_sales["avg_price_per_unit"] = (
    product_sales["total_sales_value"]
    / product_sales["units_sold"]
)

product_sales.sort_values(by="total_sales_value",ascending=False).head(10)

,units_sold,orders_with_product,avg_units_per_order,total_sales_value,avg_price_per_unit
product_id,,,,,
bb50f2e236e5eea0100680137654686c,195,187,1.042781,63885.00,327.615385
6cdd53843498f92890544667809f1595,156,151,1.033113,54730.20,350.834615
d6160fb7873f184099d9bc95e30376af,35,35,1.000000,48899.34,1397.124000
d1c427060a0f73f6b889a5c7c61f2ac4,343,323,1.061920,47214.51,137.651633
99a4788cb24856965c36a24e339b6058,488,467,1.044968,43025.56,88.167131
3dd2a17168ec895c781a9191c1e95ad7,274,255,1.074510,41082.60,149.936496
25c38557cf793876c5abdd5931f922db,38,38,1.000000,38907.32,1023.876842
5f504b3a1c75b73d6151be81eb05bdc9,63,63,1.000000,37733.90,598.950794
53b36df67ebb7c41585e8d54d6772e08,323,306,1.055556,37683.42,116.666935


In [59]:
products = datasets["products"]
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [60]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 2.3 MB


In [61]:
order_items_with_product_category = order_items.merge(
    products[["product_id", "product_category_name"]],
    on="product_id",
    how="left"
)

order_items_with_product_category.info()

order_items_with_product_category.head(25)

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 8 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   order_id               112650 non-null  str    
 1   order_item_id          112650 non-null  int64  
 2   product_id             112650 non-null  str    
 3   seller_id              112650 non-null  str    
 4   shipping_limit_date    112650 non-null  str    
 5   price                  112650 non-null  float64
 6   freight_value          112650 non-null  float64
 7   product_category_name  111047 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 6.9 MB


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,moveis_decoracao
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumaria
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,ferramentas_jardim
5,00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23 03:55:27,21.90,12.69,utilidades_domesticas
6,00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14 12:10:31,19.90,11.85,telefonia
7,000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10 12:30:45,810.00,70.75,ferramentas_jardim
8,0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26 18:31:29,145.95,11.65,beleza_saude
9,0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06 14:10:56,53.99,11.40,livros_tecnicos


In [62]:
category_sales = (
    order_items_with_product_category
    .groupby("product_category_name")
    .agg(
        units_sold=("order_item_id", "count"),
        orders=("order_id", "nunique"),
        total_sales_value=("price", "sum"),
        total_freight_value=("freight_value", "sum")
    )
    .sort_values("total_sales_value", ascending=False)
)

category_sales.head(10)

,units_sold,orders,total_sales_value,total_freight_value
product_category_name,,,,
beleza_saude,9670,8836,1258681.34,182566.73
relogios_presentes,5991,5624,1205005.68,100535.93
cama_mesa_banho,11115,9417,1036988.68,204693.04
esporte_lazer,8641,7720,988048.97,168607.51
informatica_acessorios,7827,6689,911954.32,147318.08
moveis_decoracao,8334,6449,729762.49,172749.30
cool_stuff,3796,3632,635290.85,84039.10
utilidades_domesticas,6964,5884,632248.66,146149.11
automotivo,4235,3897,592720.11,92664.21


In [63]:
order_items_with_product_category["product_category_name"] = order_items_with_product_category["product_category_name"].fillna("unknown")

category_sales = (
    order_items_with_product_category
    .groupby("product_category_name")
    .agg(
        units_sold=("order_item_id", "count"),
        orders=("order_id", "nunique"),
        total_sales_value=("price", "sum"),
        total_freight_value=("freight_value", "sum")
    )
    .sort_values("total_sales_value", ascending=False)
)

category_sales.head(10)

,units_sold,orders,total_sales_value,total_freight_value
product_category_name,,,,
beleza_saude,9670,8836,1258681.34,182566.73
relogios_presentes,5991,5624,1205005.68,100535.93
cama_mesa_banho,11115,9417,1036988.68,204693.04
esporte_lazer,8641,7720,988048.97,168607.51
informatica_acessorios,7827,6689,911954.32,147318.08
moveis_decoracao,8334,6449,729762.49,172749.30
cool_stuff,3796,3632,635290.85,84039.10
utilidades_domesticas,6964,5884,632248.66,146149.11
automotivo,4235,3897,592720.11,92664.21


In [64]:
category_sales["avg_price_per_unit"] = category_sales["total_sales_value"] / category_sales["units_sold"]
category_sales.head(10)

,units_sold,orders,total_sales_value,total_freight_value,avg_price_per_unit
product_category_name,,,,,
beleza_saude,9670,8836,1258681.34,182566.73,130.163531
relogios_presentes,5991,5624,1205005.68,100535.93,201.135984
cama_mesa_banho,11115,9417,1036988.68,204693.04,93.296327
esporte_lazer,8641,7720,988048.97,168607.51,114.344285
informatica_acessorios,7827,6689,911954.32,147318.08,116.513903
moveis_decoracao,8334,6449,729762.49,172749.30,87.564494
cool_stuff,3796,3632,635290.85,84039.10,167.357969
utilidades_domesticas,6964,5884,632248.66,146149.11,90.788148
automotivo,4235,3897,592720.11,92664.21,139.957523


In [65]:
category_sales["freight_pct_of_sales"] = category_sales["total_freight_value"] / category_sales["total_sales_value"] * 100

category_sales.sort_values(by="freight_pct_of_sales", ascending=False).head(10)

,units_sold,orders,total_sales_value,total_freight_value,avg_price_per_unit,freight_pct_of_sales
product_category_name,,,,,,
casa_conforto_2,30,24,760.27,410.31,25.342333,53.968985
flores,33,29,1110.04,488.87,33.637576,44.040755
moveis_colchao_e_estofado,38,38,4368.08,1630.46,114.949474,37.326697
artigos_de_natal,153,128,8800.82,3229.30,57.521699,36.693172
fraldas_higiene,39,27,1567.59,573.68,40.194615,36.596304
cds_dvds_musicais,14,12,730.00,224.99,52.142857,30.820548
sinalizacao_e_seguranca,199,140,21509.23,6507.82,108.086583,30.255941
alimentos_bebidas,278,227,15179.48,4507.99,54.602446,29.697921
eletronicos,2767,2550,160246.74,46578.32,57.913531,29.066626


In [66]:
sellers = datasets["sellers"]

sellers.info()
sellers.head()


<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   int64
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: int64(1), str(3)
memory usage: 96.8 KB


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [67]:
print("Total Rows :", len(sellers))
print("Unique Seller IDs :", sellers["seller_id"].nunique())

seller_state_counts = sellers["seller_state"].value_counts()
seller_state_counts



Total Rows : 3095
Unique Seller IDs : 3095


seller_state
SP    1849
PR     349
MG     244
SC     190
RJ     171
RS     129
GO      40
DF      30
ES      23
BA      19
CE      13
PE       9
PB       6
RN       5
MS       5
MT       4
RO       2
SE       2
AC       1
PI       1
MA       1
AM       1
PA       1
Name: count, dtype: int64

In [68]:
seller_state_pct = seller_state_counts / sellers["seller_id"].nunique() * 100
seller_state_pct

seller_state
SP    59.741519
PR    11.276252
MG     7.883683
SC     6.138934
RJ     5.525040
RS     4.168013
GO     1.292407
DF     0.969305
ES     0.743134
BA     0.613893
CE     0.420032
PE     0.290792
PB     0.193861
RN     0.161551
MS     0.161551
MT     0.129241
RO     0.064620
SE     0.064620
AC     0.032310
PI     0.032310
MA     0.032310
AM     0.032310
PA     0.032310
Name: count, dtype: float64

In [69]:
seller_sales = (
    order_items.groupby("seller_id")
    .agg(
        units_sold=("order_item_id", "count"),
        unique_orders=("order_id", "nunique"),
        total_sales_value=("price", "sum")
    )
    .sort_values("total_sales_value", ascending=False)
)

seller_sales.head(10)

,units_sold,unique_orders,total_sales_value
seller_id,,,
4869f7a5dfa277a7dca6462dcf3b52b2,1156,1132,229472.63
53243585a1d6dc2643021fd1853d8905,410,358,222776.05
4a3ca9315b744ce9f8e9374361493884,1987,1806,200472.92
fa1c13f2614d7b5c4749cbc52fecda94,586,585,194042.03
7c67e1448b00f6e969d365cea6b010ab,1364,982,187923.89
7e93a43ef30c4f03f38b393420bc753a,340,336,176431.87
da8622b14eb17ae2831f4ac5b9dab84a,1551,1314,160236.57
7a67c85e85bb2ce8582c35f2203ad736,1171,1160,141745.53
1025f0e2d44d7041d6cf58b6550e0bfa,1428,915,138968.55


In [70]:
seller_sales["avg_price_per_unit"] = seller_sales["total_sales_value"] / seller_sales["units_sold"]
seller_sales.head(10)

,units_sold,unique_orders,total_sales_value,avg_price_per_unit
seller_id,,,,
4869f7a5dfa277a7dca6462dcf3b52b2,1156,1132,229472.63,198.505735
53243585a1d6dc2643021fd1853d8905,410,358,222776.05,543.356220
4a3ca9315b744ce9f8e9374361493884,1987,1806,200472.92,100.892260
fa1c13f2614d7b5c4749cbc52fecda94,586,585,194042.03,331.129744
7c67e1448b00f6e969d365cea6b010ab,1364,982,187923.89,137.774113
7e93a43ef30c4f03f38b393420bc753a,340,336,176431.87,518.917265
da8622b14eb17ae2831f4ac5b9dab84a,1551,1314,160236.57,103.311779
7a67c85e85bb2ce8582c35f2203ad736,1171,1160,141745.53,121.046567
1025f0e2d44d7041d6cf58b6550e0bfa,1428,915,138968.55,97.316912


In [71]:
total_sales_value = seller_sales["total_sales_value"].sum()

top_10_seller_value = seller_sales.head(10)["total_sales_value"].sum()

top_10_sellers_value_contribution = top_10_seller_value / total_sales_value * 100

print("Total Sales:", total_sales_value)
print("Sales by top 10 sellers:", top_10_seller_value)
print("Contribution of top 10 sellers in total sales value: ", top_10_sellers_value_contribution,"%")

Total Sales: 13591643.700000001
Sales by top 10 sellers: 1787241.74
Contribution of top 10 sellers in total sales value:  13.14956291857474 %


In [72]:
seller_sales["total_sales_value"].describe()

count      3095.000000
mean       4391.484233
std       13921.997192
min           3.500000
25%         208.850000
50%         821.480000
75%        3280.830000
max      229472.630000
Name: total_sales_value, dtype: float64

Here, a crucial lesson to remember is that - Sorting decides how the rows are displayed, whereas Quantiles describe where values sit within the overall distribution. Hence, when we applied .describe() to the total_sales_value column, it shows the 25% as 208.85 and not some higher values, despite the fact that seller_sales dataframe itself is constructed by keeping total_sales_value in descending order. Sorting is only for the display purposes. When describe or quartiles are calculated, they are calculated on the overall values that are there in the dataset, regardless of how they are sorted.

In [73]:
seller_sales["sales_band"] = pd.qcut(
    seller_sales["total_sales_value"],
    q=4,
    labels=["Low", "Medium", "High", "Very High"]
)

seller_sales.tail()

,units_sold,unique_orders,total_sales_value,avg_price_per_unit,sales_band
seller_id,,,,,
34aefe746cd81b7f3b23253ea28bef39,1,1,8.0,8.0,Low
702835e4b785b67a084280efca355756,1,1,7.6,7.6,Low
1fa2d3def6adfa70e58c276bb64fe5bb,1,1,6.9,6.9,Low
77128dec4bec4878c37ab7d6169d6f26,1,1,6.5,6.5,Low
cf6f6bc4df3999b9c6440f124fb2f687,1,1,3.5,3.5,Low


In [74]:
seller_distribution = (
    seller_sales
    .groupby("sales_band")
    .agg(
        number_of_sellers = ("total_sales_value", "count"),
        sales_value_generated = ("total_sales_value", "sum"),
    )
)

seller_distribution["sales_value_share_pct"] = (seller_distribution["sales_value_generated"] / seller_distribution["sales_value_generated"].sum())*100

seller_distribution

,number_of_sellers,sales_value_generated,sales_value_share_pct
sales_band,,,
Low,774,78423.16,0.576995
Medium,774,359264.39,2.643274
High,773,1315369.95,9.677784
Very High,774,11838586.20,87.101946


In [75]:
seller_sales_with_location = seller_sales.reset_index().merge(
    sellers[["seller_id","seller_state","seller_city"]],
    on="seller_id",
    how="left"
)

seller_sales_by_state = seller_sales_with_location.groupby("seller_state").agg(unique_sellers=("seller_id","nunique"),sales_value_per_seller=("total_sales_value","sum")).sort_values(by="sales_value_per_seller",ascending=False)

seller_sales_by_state["pct_contribution"] = (seller_sales_by_state["sales_value_per_seller"] / seller_sales_by_state["sales_value_per_seller"].sum())*100

seller_sales_by_state

,unique_sellers,sales_value_per_seller,pct_contribution
seller_state,,,
SP,1849,8753396.21,64.402779
PR,349,1261887.21,9.284287
MG,244,1011564.74,7.442549
RJ,171,843984.22,6.209582
SC,190,632426.07,4.653051
RS,129,378559.54,2.785237
BA,19,285561.56,2.101008
DF,30,97749.48,0.719188
PE,9,91493.85,0.673163


The marketplace has a broad seller base, but sales value is highly concentrated among the higher-performing sellers. Geographically, São Paulo dominates both seller presence and seller-generated sales.

In [76]:
reviews = datasets["reviews"]

reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB


In [77]:
reviews.shape
reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [78]:
reviews["review_id"].nunique()

98410

In [79]:
reviews["order_id"].nunique()

98673

In [80]:
reviews["review_id"].value_counts().value_counts()

count
1    97621
2      764
3       25
Name: count, dtype: int64

In [81]:
reviews["order_id"].value_counts().value_counts()

count
1    98126
2      543
3        4
Name: count, dtype: int64

In [82]:
orders["order_id"].nunique() - reviews["order_id"].nunique(), orders["order_id"].nunique()

(768, 99441)

In [83]:
reviews["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [84]:
review_score_pct = (
    reviews["review_score"].value_counts(normalize=True)
    .sort_index()
    * 100
)

review_score_pct

review_score
1    11.513344
2     3.175643
3     8.242965
4    19.291704
5    57.776344
Name: proportion, dtype: float64

In [85]:
multiple_review_orders = (
    reviews["order_id"]
    .value_counts()
)

multiple_review_orders = multiple_review_orders[multiple_review_orders > 1]
multiple_review_orders

order_id
c88b1d1b157a9999ce368f218a407141    3
df56136b8031ecd28e200bb18e6ddb2e    3
03c939fd7fd3b38f8485a0f95798f1f6    3
8e17072ec97ce29f0e1f111e598b0c85    3
cf73e2cb1f4a9480ed70c154da3d954a    2
                                   ..
e00ed9d20c3479f9f0e9727ca9d60946    2
5c2049caca936695cfa43805ebe7d8e3    2
6066fbcebeaf378770ad9fbe8e30d1f0    2
2739713764165279791b394484947e2d    2
157de1652c032429a625d034be79eeeb    2
Name: count, Length: 547, dtype: int64

In [86]:
three_review_orders = multiple_review_orders[multiple_review_orders == 3]

reviews[reviews["order_id"].isin(three_review_orders.index)].sort_values("order_id")

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
8273,b04ed893318da5b863e878cd3d0511df,03c939fd7fd3b38f8485a0f95798f1f6,3,NaN,Um ponto negativo que achei foi a cobrança de ...,2018-03-20 00:00:00,2018-03-21 02:28:23
51527,f4bb9d6dd4fb6dcc2298f0e7b17b8e1e,03c939fd7fd3b38f8485a0f95798f1f6,4,NaN,NaN,2018-03-29 00:00:00,2018-03-30 00:29:09
69438,405eb2ea45e1dbe2662541ae5b47e2aa,03c939fd7fd3b38f8485a0f95798f1f6,3,NaN,Seria ótimo se tivesem entregue os 3 (três) pe...,2018-03-06 00:00:00,2018-03-06 19:50:32
44694,67c2557eb0bd72e3ece1e03477c9dff5,8e17072ec97ce29f0e1f111e598b0c85,1,NaN,Entregou o produto errado.,2018-04-07 00:00:00,2018-04-08 22:48:27
64510,2d6ac45f859465b5c185274a1c929637,8e17072ec97ce29f0e1f111e598b0c85,1,NaN,Comprei 3 unidades do produto vieram 2 unidade...,2018-04-07 00:00:00,2018-04-07 21:13:05
92300,6e4c4086d9611ae4cc0cc65a262751fe,8e17072ec97ce29f0e1f111e598b0c85,1,NaN,"Embora tenha entregue dentro do prazo, não env...",2018-04-14 00:00:00,2018-04-16 11:37:31
1985,ffb8cff872a625632ac983eb1f88843c,c88b1d1b157a9999ce368f218a407141,3,NaN,NaN,2017-07-22 00:00:00,2017-07-26 13:41:07
82525,202b5f44d09cd3cfc0d6bd12f01b044c,c88b1d1b157a9999ce368f218a407141,5,NaN,NaN,2017-07-22 00:00:00,2017-07-26 13:40:22
89360,fb96ea2ef8cce1c888f4d45c8e22b793,c88b1d1b157a9999ce368f218a407141,5,NaN,NaN,2017-07-21 00:00:00,2017-07-26 13:45:15
2952,c444278834184f72b1484dfe47de7f97,df56136b8031ecd28e200bb18e6ddb2e,5,NaN,NaN,2017-02-08 00:00:00,2017-02-14 13:58:48


Some orders have multiple review records, and the three-review examples suggest that multiple products/units within an order may be related to this behavior. However, the reviews table does not contain order_item_id or product_id, so we cannot establish a one-to-one relationship between reviews and individual items.

In [87]:
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [88]:
multi_review_order_items = order_items[order_items["order_id"].isin(multiple_review_orders.index)]

multi_review_order_summary = (
    multi_review_order_items
    .groupby("order_id")
    .agg(
        order_item_count=("order_item_id", "nunique"),
        product_count=("product_id", "nunique")
    )
)

multi_review_order_summary["review_count"] = multiple_review_orders
multi_review_order_summary[multi_review_order_summary["review_count"] > 1].head(20)

,order_item_count,product_count,review_count
order_id,,,
0035246a40f520710769010f752e7507,1,1,2
013056cfe49763c6f66bda03396c5ee3,1,1,2
0176a6846bcb3b0d3aa3116a9a768597,1,1,2
02355020fd0a40a0d56df9f6ff060413,1,1,2
029863af4b968de1e5d6a82782e662f5,1,1,2
02e0b68852217f5715fb9cc885829454,1,1,2
03515a836bb855b03f7df9dee520a8fc,1,1,2
03c939fd7fd3b38f8485a0f95798f1f6,1,1,3
03eba6d9fef8f5b3e811d4b5a7cca9cd,1,1,2


Review table grain investigation: Most orders have a single review record, but 547 orders contain multiple review records. While comparing these orders with order_items, no consistent relationship between multiple reviews and multiple order items/products is found. Therefore, multiple reviews cannot be reliably interpreted as item-level reviews using the available data.

In [89]:
single_review_orders = (
    reviews["order_id"]
    .value_counts()
)

single_review_orders = single_review_orders[
    single_review_orders == 1
].index

reviews_one_per_order = reviews[reviews["order_id"].isin(single_review_orders)].copy()

reviews_one_per_order["order_id"].isin(orders["order_id"]).value_counts()

order_id
True    98126
Name: count, dtype: int64

Earlier we had 98,673 unique reviewed orders overall. The difference is:
98,673 − 98,126 = 547. Now those are precisely the no. of orders with multiple review records that we've decided to exclude from this particular order-level analysis. Also, this check was performed to make sure that there are no orders with no reviews given.

In [90]:
reviews_and_orders = reviews_one_per_order.merge(
    orders[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "delivery_delay_days",
            "delivery_delay_type"
        ]
    ],
    on="order_id",
    how="inner"
)

reviews_and_orders.info()


<class 'pandas.DataFrame'>
RangeIndex: 98126 entries, 0 to 98125
Data columns (total 13 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   review_id                      98126 non-null  str           
 1   order_id                       98126 non-null  str           
 2   review_score                   98126 non-null  int64         
 3   review_comment_title           11534 non-null  str           
 4   review_comment_message         40592 non-null  str           
 5   review_creation_date           98126 non-null  str           
 6   review_answer_timestamp        98126 non-null  str           
 7   order_status                   98126 non-null  str           
 8   order_purchase_timestamp       98126 non-null  datetime64[us]
 9   order_delivered_customer_date  95305 non-null  datetime64[us]
 10  order_estimated_delivery_date  98126 non-null  datetime64[us]
 11  delivery_delay_days       

In [91]:
reviews_and_orders[reviews_and_orders["order_delivered_customer_date"].isna()]["order_status"].value_counts()

order_status
shipped        1021
canceled        595
unavailable     593
invoiced        305
processing      294
delivered         8
created           3
approved          2
Name: count, dtype: int64

In [92]:
reviews_and_orders.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay_days,delivery_delay_type
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59,delivered,2018-01-11 15:30:49,2018-01-17 18:42:41,2018-02-02,-16.0,Early
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13,delivered,2018-02-28 12:25:19,2018-03-09 23:17:20,2018-03-14,-5.0,Early
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24,delivered,2018-02-03 09:56:22,2018-02-16 17:28:48,2018-03-09,-21.0,Early
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06,delivered,2017-04-09 17:41:13,2017-04-20 09:08:35,2017-05-10,-20.0,Early
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53,delivered,2018-02-10 10:59:03,2018-02-28 16:33:35,2018-03-09,-9.0,Early


In [93]:
reviews_and_orders["delivery_delay_type"].value_counts()
reviews_and_orders.groupby("delivery_delay_type")["review_score"].agg(["count", "mean"])

,count,mean
delivery_delay_type,,
Early,87682,4.294690
Late,6354,2.272742
On time,1269,4.037825


In [94]:
delivery_review_distribution = pd.crosstab(
    reviews_and_orders["delivery_delay_type"],
    reviews_and_orders["review_score"],
    normalize="index"
) * 100

delivery_review_distribution

review_score,1,2,3,4,5
delivery_delay_type,,,,,
Early,6.580598,2.627677,7.977692,20.370201,62.443831
Late,53.745672,8.608750,10.859301,10.198300,16.587976
On time,8.431836,3.940110,13.869188,22.931442,50.827423


Late deliveries are strongly associated with poor customer satisfaction. Among orders delivered early, 62.4% received 5-star reviews and only 6.6% received 1-star reviews. For late deliveries, the pattern almost reverses: 53.7% received 1-star reviews while only 16.6% received 5-star reviews.

In [95]:
late_orders = reviews_and_orders[
    reviews_and_orders["delivery_delay_type"] == "Late"
].copy()

late_orders["delivery_delay_days"].describe()

count    6354.000000
mean       10.510387
std        14.477879
min         1.000000
25%         3.000000
50%         7.000000
75%        13.000000
max       188.000000
Name: delivery_delay_days, dtype: float64

Next - Does customer satisfaction deteriorate as the delivery delay becomes more severe?

In [96]:
late_orders["delay_severity"] = pd.cut(
    late_orders["delivery_delay_days"],
    bins=[0, 3, 7, 14, float("inf")],
    labels=[
        "Slightly late",
        "Moderately late",
        "Significantly late",
        "Severely late"
    ]
)

late_orders["delay_severity"].value_counts().sort_index()

delay_severity
Slightly late         1848
Moderately late       1740
Significantly late    1441
Severely late         1325
Name: count, dtype: int64

In [97]:
delay_severity_review_distribution = pd.crosstab(
    late_orders["delay_severity"],
    late_orders["review_score"],
    normalize="index"
) * 100

delay_severity_review_distribution

review_score,1,2,3,4,5
delay_severity,,,,,
Slightly late,25.108225,7.034632,14.610390,20.021645,33.225108
Moderately late,58.505747,9.022989,9.885057,8.448276,14.137931
Significantly late,70.784178,9.437890,8.813324,4.163775,6.800833
Severely late,68.905660,9.358491,9.132075,5.358491,7.245283


Next Q - Are certain product categories associated with lower customer satisfaction?

In [98]:
order_items_with_product_category.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,moveis_decoracao
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumaria
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,ferramentas_jardim


In [99]:
product_categories_per_orderid = order_items_with_product_category.groupby("order_id")["product_category_name"].nunique()
product_categories_per_orderid.describe()

count    98666.000000
mean         1.008149
std          0.091909
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          3.000000
Name: product_category_name, dtype: float64

In [100]:
product_categories_per_orderid.value_counts().sort_index()

product_category_name
1    97880
2      768
3       18
Name: count, dtype: int64

In [101]:
single_category_orders = product_categories_per_orderid[product_categories_per_orderid == 1].index
single_category_orders

Index(['00010242fe8c5a6d1ba2dd792cb16214', '00018f77f2f0320c557190d7a144bdd3',
       '000229ec398224ef6ca0657da4fc703e', '00024acbcdf0a6daa1e931b038114c75',
       '00042b26cf59d7ce69dfabb4e55b4fd9', '00048cc3ae777c65dbb7d2a0634bc1ea',
       '00054e8431b9d7675808bcb819fb4a32', '000576fe39319847cbb9d288c5617fa6',
       '0005a1a1728c9d785b8e2b08b904576c', '0005f50442cb953dcd1d21e1fb923495',
       ...
       'fffa82886406ccf10c7b4e35c4ff2788', 'fffb0b1a50e65c449020434fa835e078',
       'fffb2ef8874127f75b52b643880fd7e0', 'fffb9224b6fc7c43ebb0904318b10b5f',
       'fffbee3b5462987e66fb49b1c5411df2', 'fffc94f6ce00a00581880bf54a75a037',
       'fffcd46ef2263f404302a634eb57f7eb', 'fffce4705a9662cd70adb13d4a31832d',
       'fffe18544ffabc95dfada21779c9644f', 'fffe41c64501cc87c801fd61db3f6244'],
      dtype='str', name='order_id', length=97880)

In [102]:
single_category_order_items = order_items_with_product_category[order_items_with_product_category["order_id"].isin(single_category_orders)][["order_id","product_category_name"]].drop_duplicates("order_id")

len(single_category_order_items), single_category_order_items["order_id"].nunique()
#single_category_order_items["order_id"].value_counts()

#single_category_order_items.loc[single_category_order_items["order_id"] == "8272b63d03f5f79c56e9e4120aec44ef"]

(97880, 97880)

In [103]:
reviews_one_per_order.head(20)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53
5,15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,NaN,NaN,2018-04-13 00:00:00,2018-04-16 00:39:37
6,07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,NaN,NaN,2017-07-16 00:00:00,2017-07-18 19:30:34
7,7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,NaN,NaN,2018-08-14 00:00:00,2018-08-14 21:36:06
8,a3f6f7f6f433de0aefbb97da197c554c,9c214ac970e84273583ab523dfafd09b,5,NaN,NaN,2017-05-17 00:00:00,2017-05-18 12:05:37
9,8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelh...,2018-05-22 00:00:00,2018-05-23 16:45:47


In [104]:
category_reviews_single = reviews_one_per_order.merge(
    single_category_order_items[["order_id", "product_category_name"]],
    on="order_id",
    how="inner"
)

In [105]:
category_reviews_single.info()

<class 'pandas.DataFrame'>
RangeIndex: 96602 entries, 0 to 96601
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                96602 non-null  str  
 1   order_id                 96602 non-null  str  
 2   review_score             96602 non-null  int64
 3   review_comment_title     11286 non-null  str  
 4   review_comment_message   39588 non-null  str  
 5   review_creation_date     96602 non-null  str  
 6   review_answer_timestamp  96602 non-null  str  
 7   product_category_name    96602 non-null  str  
dtypes: int64(1), str(7)
memory usage: 5.9 MB


In [106]:
category_reviews_single["order_id"].nunique()

96602

In [107]:
category_reviews_single.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,product_category_name
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59,esporte_lazer
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13,informatica_acessorios
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24,informatica_acessorios
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06,ferramentas_jardim
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53,esporte_lazer


In [108]:
category_review_summary = (
    category_reviews_single
    .groupby("product_category_name")["review_score"]
    .agg(["count", "mean"])
    .sort_values("mean")
)

category_review_summary

,count,mean
product_category_name,,
seguros_e_servicos,2,2.500000
pc_gamer,7,3.428571
portateis_cozinha_e_preparadores_de_alimentos,14,3.428571
moveis_escritorio,1243,3.630732
fashion_roupa_masculina,110,3.727273
...,...,...
construcao_ferramentas_ferramentas,94,4.425532
livros_interesse_geral,503,4.467197
flores,26,4.538462


In [109]:
category_review_summary_100 = category_review_summary[
    category_review_summary["count"] >= 100
].sort_values("mean")

category_review_summary_100

,count,mean
product_category_name,,
moveis_escritorio,1243,3.630732
fashion_roupa_masculina,110,3.727273
audio,340,3.852941
construcao_ferramentas_seguranca,159,3.874214
telefonia_fixa,211,3.890995
fashion_underwear_e_moda_praia,120,3.933333
casa_conforto,344,3.947674
unknown,1370,3.955474
cama_mesa_banho,8997,3.991664


In [110]:
lowest_5_categories = category_review_summary_100.head(5).index
lowest_5_categories

Index(['moveis_escritorio', 'fashion_roupa_masculina', 'audio',
       'construcao_ferramentas_seguranca', 'telefonia_fixa'],
      dtype='str', name='product_category_name')

In [111]:
category_review_distribution = pd.crosstab(
    category_reviews_single[
        category_reviews_single["product_category_name"].isin(lowest_5_categories)
    ]["product_category_name"],
    category_reviews_single[
        category_reviews_single["product_category_name"].isin(lowest_5_categories)
    ]["review_score"],
    normalize="index"
) * 100
category_review_distribution

review_score,1,2,3,4,5
product_category_name,,,,,
audio,16.176471,5.000000,9.705882,15.588235,53.529412
construcao_ferramentas_seguranca,16.981132,2.515723,8.176101,20.754717,51.572327
fashion_roupa_masculina,22.727273,2.727273,6.363636,15.454545,52.727273
moveis_escritorio,17.779566,4.585680,14.320193,23.411102,39.903459
telefonia_fixa,15.639810,3.317536,8.530806,21.327014,51.184834


In [112]:
category_reviews_single["review_score"].mean()

np.float64(4.1137968158009155)

In [113]:
category_review_summary_100.tail(5)

,count,mean
product_category_name,,
alimentos,437,4.281465
malas_acessorios,1015,4.340887
alimentos_bebidas,221,4.380090
livros_tecnicos,251,4.414343
livros_interesse_geral,503,4.467197


Customer satisfaction varies considerably across product categories. Among categories with at least 100 reviewed orders, moveis_escritorio had the lowest average rating (3.63), while livros_interesse_geral had the highest (4.47), compared with an overall average of 4.11.

Next Q - Does order value have any meaningful relationship with customer satisfaction?

In [114]:
order_value.columns


Index(['total_product_value', 'total_freight_value', 'item_count',
       'freight_pct_of_product_value'],
      dtype='str')

In [115]:
order_value.index.nunique(), len(order_value)

(98666, 98666)

In [116]:
reviews_order_value = reviews_one_per_order.merge(
    order_value[["total_product_value"]],
    left_on="order_id",
    right_index=True,
    how="inner"
)

reviews_order_value.info()

<class 'pandas.DataFrame'>
Index: 97373 entries, 0 to 99223
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   review_id                97373 non-null  str    
 1   order_id                 97373 non-null  str    
 2   review_score             97373 non-null  int64  
 3   review_comment_title     11445 non-null  str    
 4   review_comment_message   40070 non-null  str    
 5   review_creation_date     97373 non-null  str    
 6   review_answer_timestamp  97373 non-null  str    
 7   total_product_value      97373 non-null  float64
dtypes: float64(1), int64(1), str(6)
memory usage: 6.7 MB


In [117]:
reviews_one_per_order["order_id"].isin(order_value.index).value_counts()

order_id
True     97373
False      753
Name: count, dtype: int64

In [118]:
reviews_one_per_order.info()

<class 'pandas.DataFrame'>
Index: 98126 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                98126 non-null  str  
 1   order_id                 98126 non-null  str  
 2   review_score             98126 non-null  int64
 3   review_comment_title     11534 non-null  str  
 4   review_comment_message   40592 non-null  str  
 5   review_creation_date     98126 non-null  str  
 6   review_answer_timestamp  98126 non-null  str  
dtypes: int64(1), str(6)
memory usage: 6.0 MB


In [119]:
missing_order_ids = reviews_one_per_order.loc[
    ~reviews_one_per_order["order_id"].isin(order_value.index),
    "order_id"
]

missing_order_ids.isin(orders["order_id"]).value_counts()

order_id
True    753
Name: count, dtype: int64

In [120]:
missing_order_ids.isin(order_items["order_id"]).value_counts()

order_id
False    753
Name: count, dtype: int64

Since the 753 orders exist in orders but have no order_items, we'd like to know what their statuses are.

In [121]:
orders.loc[
    orders["order_id"].isin(missing_order_ids),
    "order_status"
].value_counts()

order_status
unavailable    587
canceled       160
created          3
invoiced         2
shipped          1
Name: count, dtype: int64

These orders never reached the normal order-item stage, so there is no product-level value from which we could construct total_product_value. Hence they are present in the orders table, but absent from order items and subsequently created dfs.

In [122]:
reviews_order_value["total_product_value"].describe()

count    97373.000000
mean       137.686227
std        209.841361
min          0.850000
25%         45.900000
50%         86.900000
75%        149.900000
max      13440.000000
Name: total_product_value, dtype: float64

In [123]:
reviews_order_value["order_value_band"] = pd.qcut(
    reviews_order_value["total_product_value"],
    q=4,
    labels=["Low", "Medium", "High", "Very High"]
)

reviews_order_value["order_value_band"].value_counts()

order_value_band
Low          24430
High         24394
Medium       24315
Very High    24234
Name: count, dtype: int64

In [124]:
reviews_order_value.groupby("order_value_band")["review_score"].agg(["count", "mean"]
)

,count,mean
order_value_band,,
Low,24430,4.170774
Medium,24315,4.131524
High,24394,4.112856
Very High,24234,4.006066


In [125]:
pd.crosstab(
    reviews_order_value["order_value_band"],
    reviews_order_value["review_score"],
    normalize="index"
) * 100

review_score,1,2,3,4,5
order_value_band,,,,,
Low,9.287761,2.857143,8.726975,19.746214,59.381907
Medium,10.240592,3.162657,8.213037,19.971211,58.412503
High,10.916619,3.054030,8.133147,19.619579,58.276625
Very High,13.728646,3.449699,7.881489,18.366757,56.573409


Higher-value orders show slightly lower customer satisfaction, primarily reflected in a higher share of 1-star reviews. However, the relationship is considerably weaker than the relationship between delivery performance and customer satisfaction.

Next Q - Does the seller have a meaningful relationship with customer satisfaction?

In [126]:
order_items_with_product_category.columns

Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value',
       'product_category_name'],
      dtype='str')

In [127]:
order_seller = order_items_with_product_category[["order_id", "seller_id"]].drop_duplicates()

order_seller["order_id"].nunique(), len(order_seller)

(98666, 100010)

In [128]:
order_seller_count = order_seller.groupby("order_id")["seller_id"].nunique()

order_seller_count.value_counts().sort_index()

seller_id
1    97388
2     1219
3       54
4        3
5        2
Name: count, dtype: int64

Note: Here, as we can see - there are certain orders which have multiple sellers. Now, ideally we should consider all the orders regardless of whether they have multiple sellers or single sellers. But since in our case there is no information present about the reviews belonging to which particular product or item, it is not possible for us to attribute the given review by a customer to a particular item present under a particular order id. Reviews dataset doesn't have product level granularity present. And since the multi seller orders are very less, approx. 1.3% of the entire population, for the sake of simplicity and given the nature of data, we have decided to exclude the order ids that have multiple sellers present under them.

In [129]:
single_seller_orders = order_seller_count[order_seller_count == 1].index

order_seller_single = order_seller[
    order_seller["order_id"].isin(single_seller_orders)
]

order_seller_single.shape

(97388, 2)

In [130]:
seller_reviews = reviews_one_per_order.merge(
    order_seller_single,
    on="order_id",
    how="inner"
)

seller_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 96122 entries, 0 to 96121
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                96122 non-null  str  
 1   order_id                 96122 non-null  str  
 2   review_score             96122 non-null  int64
 3   review_comment_title     11175 non-null  str  
 4   review_comment_message   39252 non-null  str  
 5   review_creation_date     96122 non-null  str  
 6   review_answer_timestamp  96122 non-null  str  
 7   seller_id                96122 non-null  str  
dtypes: int64(1), str(7)
memory usage: 5.9 MB


In [131]:
reviews_one_per_order["order_id"].isin(order_seller_single["order_id"]).value_counts()

order_id
True     96122
False     2004
Name: count, dtype: int64

In [132]:
missing_seller_orders = reviews_one_per_order[~reviews_one_per_order["order_id"].isin(order_seller_single["order_id"])]

missing_seller_orders["order_id"].isin(orders["order_id"]).value_counts()

order_id
True    2004
Name: count, dtype: int64

In [133]:
missing_seller_orders["order_id"].isin(order_items["order_id"]).value_counts()

order_id
True     1251
False     753
Name: count, dtype: int64

In [134]:
missing_seller_orders[
    missing_seller_orders["order_id"].isin(order_items["order_id"])
]["order_id"].isin(
        order_seller_count[order_seller_count > 1].index
).value_counts()

order_id
True    1251
Name: count, dtype: int64

Note - Out of 98126 orders that are there in the reviews_one_per_order, only 96122 were included in the merge result. The remaining 2004 orders include 1251 multi seller orders that are rightly removed from the result as it is an inner join. And the rest 753 orders are the same orders that are having no entry into order items table and are hence missing from the merge result.

In [135]:
seller_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,seller_id
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59,6d803cb79cc31c41c4c789a75933b3c7
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13,8e6d7754bc7e0f22c96d255ebda59eba
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24,a1043bafd471dff536d0c462352beb48
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06,a1043bafd471dff536d0c462352beb48
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53,989becdce12ebc39863c2bceab6f3ca1


In [136]:
seller_review_count = (
    seller_reviews
    .groupby("seller_id")["review_score"]
    .count()
    .sort_values(ascending=False)
)

seller_review_count

seller_id
6560211a19b47992c3666cc44a7e94c0    1801
4a3ca9315b744ce9f8e9374361493884    1672
cc419e0650a3c5ba77189a1882b7556a    1652
1f50f920176fa81dab994f9023523100    1359
955fee9216a65b617aa5c0531780ce60    1266
                                    ... 
f9eda05b67bef472deaddbba84aca289       1
a09a13f18ed27932d08438aa6f12756e       1
5e063e85d44b0f5c3e6ec3131103a57e       1
05a48cc8859962767935ab9087417fbb       1
fc0b214b59a83615fea981c6424a02ae       1
Name: review_score, Length: 3069, dtype: int64

In [137]:
seller_review_count.describe()

count    3069.00000
mean       31.32030
std       101.69223
min         1.00000
25%         2.00000
50%         6.00000
75%        21.00000
max      1801.00000
Name: review_score, dtype: float64

In [138]:
(seller_review_count >= 10).value_counts()

review_score
False    1826
True     1243
Name: count, dtype: int64

In [139]:
seller_rating_summary = (
    seller_reviews
    .groupby("seller_id")["review_score"]
    .agg(["count", "mean"])
)

seller_rating_summary

,count,mean
seller_id,,
0015a82c2db000af6aaaf3ae2ecb0532,3,3.666667
001cca7ae9ae17fb1caed9dfb1094831,196,4.000000
001e6ad469a905060d959994f1b41e4f,1,1.000000
002100f778ceb8431b7a1020ff7ab48f,48,3.937500
003554e2dce176b5555353e4f3555ac8,1,5.000000
...,...,...
ffcfefa19b08742c5d315f2791395ee5,1,1.000000
ffdd9f82b9a447f6f8d4b91554cc7dd3,18,4.333333
ffeee66ac5d5a62fe688b9d26f83f534,14,4.214286


In [140]:
seller_rating_summary["mean"].describe()

count    3069.000000
mean        4.036860
std         0.953568
min         1.000000
25%         3.833333
50%         4.238245
75%         4.666667
max         5.000000
Name: mean, dtype: float64

In [141]:
seller_rating_summary["count"].corr(seller_rating_summary["mean"])

np.float64(0.027454793784117613)

In [142]:
seller_rating_summary["review_volume_band"] = pd.cut(
    seller_rating_summary["count"],
    bins=[0, 5, 20, 50, float("inf")],
    labels=["1-5", "6-20", "21-50", "51+"]
)

seller_rating_summary["review_volume_band"].value_counts().sort_index()

review_volume_band
1-5      1453
6-20      845
21-50     364
51+       407
Name: count, dtype: int64

In [143]:
seller_rating_summary.groupby("review_volume_band")["mean"].agg(
    ["count", "mean"]
)

seller_rating_summary

,count,mean,review_volume_band
seller_id,,,
0015a82c2db000af6aaaf3ae2ecb0532,3,3.666667,1-5
001cca7ae9ae17fb1caed9dfb1094831,196,4.000000,51+
001e6ad469a905060d959994f1b41e4f,1,1.000000,1-5
002100f778ceb8431b7a1020ff7ab48f,48,3.937500,21-50
003554e2dce176b5555353e4f3555ac8,1,5.000000,1-5
...,...,...,...
ffcfefa19b08742c5d315f2791395ee5,1,1.000000,1-5
ffdd9f82b9a447f6f8d4b91554cc7dd3,18,4.333333,6-20
ffeee66ac5d5a62fe688b9d26f83f534,14,4.214286,6-20


Sellers with only 1–5 reviewed orders have a substantially lower average seller rating (3.91) than sellers with 6+ reviewed orders (roughly 4.14–4.17).

In [144]:
seller_rating_6plus = seller_rating_summary[seller_rating_summary["count"] >= 6].sort_values("mean")

seller_rating_6plus.head(10)

,count,mean,review_volume_band
seller_id,,,
4342d4b2ba6b161468c63a7e7cfce593,19,1.263158,6-20
30c7f28fd3a5897b2c82d152bb760c17,6,1.666667,6-20
b1b3948701c5c72445495bd161b83a4c,18,1.722222,6-20
5151aea44289d6c6b090ee31c2132508,6,1.833333,6-20
8bd0e3abda539b9479c4b44a691be1ec,6,1.833333,6-20
64b87978a11c1ea7501a89eabe8c2c1a,7,2.000000,6-20
4b1eaadf791bdbbad8c4a35b65236d52,8,2.000000,6-20
cf8ab1616079e2793aa29d524df01bb1,9,2.000000,6-20
ffff564a4f9085cd26170f4732393726,20,2.100000,6-20


In [145]:
seller_rating_6plus["one_star_count"] = (
    seller_reviews[seller_reviews["review_score"] == 1]
    .groupby("seller_id")
    .size()
)

seller_rating_6plus["one_star_count"] = (
    seller_rating_6plus["one_star_count"]
    .fillna(0)
    .astype(int)
)

seller_rating_6plus.head(10)

,count,mean,review_volume_band,one_star_count
seller_id,,,,
4342d4b2ba6b161468c63a7e7cfce593,19,1.263158,6-20,17
30c7f28fd3a5897b2c82d152bb760c17,6,1.666667,6-20,4
b1b3948701c5c72445495bd161b83a4c,18,1.722222,6-20,14
5151aea44289d6c6b090ee31c2132508,6,1.833333,6-20,4
8bd0e3abda539b9479c4b44a691be1ec,6,1.833333,6-20,4
64b87978a11c1ea7501a89eabe8c2c1a,7,2.000000,6-20,4
4b1eaadf791bdbbad8c4a35b65236d52,8,2.000000,6-20,5
cf8ab1616079e2793aa29d524df01bb1,9,2.000000,6-20,6
ffff564a4f9085cd26170f4732393726,20,2.100000,6-20,13


In [146]:
seller_reviews["review_score"].eq(1).sum()

np.int64(10299)

In [147]:
seller_rating_6plus.head(10)["one_star_count"].sum()

np.int64(77)

Next Q - Does the payment method or payment structure have any meaningful relationship with customer satisfaction?

In [148]:
payments.columns

Index(['order_id', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value'],
      dtype='str')

In [149]:
payments["order_id"].value_counts().value_counts().sort_index()

count
1     96479
2      2382
3       301
4       108
5        52
6        36
7        28
8        11
9         9
10        5
11        8
12        8
13        3
14        2
15        2
19        2
21        1
22        1
26        1
29        1
Name: count, dtype: int64

In [150]:
payment_type_count = (
    payments.groupby("order_id")["payment_type"]
    .nunique()
)

payment_type_count.value_counts()

payment_type
1    97194
2     2246
Name: count, dtype: int64

In [151]:
payment_type_count.head()

order_id
00010242fe8c5a6d1ba2dd792cb16214    1
00018f77f2f0320c557190d7a144bdd3    1
000229ec398224ef6ca0657da4fc703e    1
00024acbcdf0a6daa1e931b038114c75    1
00042b26cf59d7ce69dfabb4e55b4fd9    1
Name: payment_type, dtype: int64

In [152]:
payments["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [153]:
order_payment_type = payment_type_count.reset_index(name="payment_type_count")

import numpy as np

order_payment_type.head()


,order_id,payment_type_count
0,00010242fe8c5a6d1ba2dd792cb16214,1
1,00018f77f2f0320c557190d7a144bdd3,1
2,000229ec398224ef6ca0657da4fc703e,1
3,00024acbcdf0a6daa1e931b038114c75,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1


In [154]:
order_payment_type["payment_type"] = np.where(
    order_payment_type["payment_type_count"] == 1,
    payments.groupby("order_id")["payment_type"].first(),
    "multiple"
)

order_payment_type["payment_type"].value_counts()

payment_type
credit_card    74259
boleto         19784
multiple        2246
voucher         1621
debit_card      1527
not_defined        3
Name: count, dtype: int64

In [155]:
reviews_payment = reviews_one_per_order.merge(
    order_payment_type,
    on="order_id",
    how="inner"
)

reviews_payment.info()

<class 'pandas.DataFrame'>
RangeIndex: 98125 entries, 0 to 98124
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                98125 non-null  str  
 1   order_id                 98125 non-null  str  
 2   review_score             98125 non-null  int64
 3   review_comment_title     11534 non-null  str  
 4   review_comment_message   40591 non-null  str  
 5   review_creation_date     98125 non-null  str  
 6   review_answer_timestamp  98125 non-null  str  
 7   payment_type_count       98125 non-null  int64
 8   payment_type             98125 non-null  str  
dtypes: int64(2), str(7)
memory usage: 6.7 MB


In [156]:
reviews_payment.groupby("payment_type")["review_score"].agg(
    ["count", "mean"]
).sort_values("mean")

,count,mean
payment_type,,
not_defined,3,1.666667
voucher,1588,3.948992
multiple,2215,4.072235
boleto,19511,4.087694
credit_card,73294,4.088834
debit_card,1514,4.171731


In [157]:
pd.crosstab(
    index = reviews_payment["payment_type"],
    columns = reviews_payment["review_score"],
    normalize="index"
) * 100

review_score,1,2,3,4,5
payment_type,,,,,
boleto,11.285941,3.121316,8.538773,19.645328,57.408641
credit_card,11.546648,3.146233,8.102983,19.285344,57.918793
debit_card,9.907530,3.038309,7.661823,18.758256,60.634082
multiple,11.060948,3.431151,9.209932,19.819413,56.478555
not_defined,66.666667,0.000000,33.333333,0.000000,0.000000
voucher,14.042821,4.596977,9.445844,16.246851,55.667506


The reason we have used normalize = "index" above is because that gives us the info that out of all the credit card payments that were made, what percentage of those payments received what review score.

In [158]:
reviews_payment[reviews_payment["payment_type"] == "voucher"]["review_score"].value_counts().sort_index()

review_score
1    223
2     73
3    150
4    258
5    884
Name: count, dtype: int64

In [159]:
payment_one_star = (
    reviews_payment[reviews_payment["review_score"] == 1]
    .groupby("payment_type")
    .size()
    .sort_values(ascending=False)
)
payment_one_star

payment_type
credit_card    8463
boleto         2202
multiple        245
voucher         223
debit_card      150
not_defined       2
dtype: int64

In [167]:
reviews_payment.head(15)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,payment_type_count,payment_type
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59,1,credit_card
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13,1,credit_card
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24,1,credit_card
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06,1,credit_card
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53,1,credit_card
5,15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,NaN,NaN,2018-04-13 00:00:00,2018-04-16 00:39:37,1,credit_card
6,07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,NaN,NaN,2017-07-16 00:00:00,2017-07-18 19:30:34,1,credit_card
7,7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,NaN,NaN,2018-08-14 00:00:00,2018-08-14 21:36:06,1,credit_card
8,a3f6f7f6f433de0aefbb97da197c554c,9c214ac970e84273583ab523dfafd09b,5,NaN,NaN,2017-05-17 00:00:00,2017-05-18 12:05:37,1,credit_card
9,8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelh...,2018-05-22 00:00:00,2018-05-23 16:45:47,1,credit_card


In [168]:
reviews_payment_summary = reviews_payment.groupby("payment_type")["review_score"].agg(reviewed_orders = "count", avg_rating="mean")
reviews_payment_summary["one_star_count"] = payment_one_star

reviews_payment_summary["one_star_rate"] = (
    reviews_payment_summary["one_star_count"]
    / reviews_payment_summary["reviewed_orders"]
    * 100
)

reviews_payment_summary = reviews_payment_summary.sort_values(by="avg_rating")

reviews_payment_summary

,reviewed_orders,avg_rating,one_star_count,one_star_rate
payment_type,,,,
not_defined,3,1.666667,2,66.666667
voucher,1588,3.948992,223,14.042821
multiple,2215,4.072235,245,11.060948
boleto,19511,4.087694,2202,11.285941
credit_card,73294,4.088834,8463,11.546648
debit_card,1514,4.171731,150,9.907530


## Conclusion from the above analysis for finding a relationship between payment types and review scores:
- Payment type shows some variation in customer satisfaction, but the differences are relatively modest compared with the stronger relationship observed between delivery performance and review scores.
- Voucher payments had the lowest average rating among the meaningful payment groups (3.95) and the highest 1-star review rate (14.04%).
- Debit card payments had the highest average rating among the meaningful payment groups (4.17) and the lowest 1-star review rate (9.91%).
- Credit card payments accounted for the largest number of 1-star reviews (8,463), but this is primarily because credit card was by far the most common payment type. Its 1-star rate was 11.55%.
- The results indicate an association between payment type and review score, but they do not establish that payment type itself causes lower or higher customer satisfaction.
- Given the relatively modest differences between the major payment groups, payment type will be treated as a secondary factor rather than a primary driver of customer dissatisfaction.

## Summary of the analysis so far

Question is - Which factors associated with an order/customer are related to customer satisfaction (review score), and can those factors eventually be used for a risk score?

1. Delivery performance -> Strong relationship with customer satisfaction -> Important factor
2. Product category -> Moderate relationship with customer satisfaction -> Potential factor
3. Order value -> Mild/moderate relationship with customer satisfaction -> Potential secondary factor
4. Seller -> Not strongly concentrated relationship with customer satisfaction -> Don't prioritize
5. Payment type -> Mild/moderate relationship with customer satisfaction -> Secondary factor

The end product is going to be a business-oriented risk framework that says - "Based on the characteristics of an order, how likely is this order to be associated with customer dissatisfaction, and which orders should the business pay attention to?"

In [169]:
customers.head(10)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
5,879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC
6,fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,4534,sao paulo,SP
7,5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,timoteo,MG
8,5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR
9,4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,belo horizonte,MG


In [170]:
order_value.head(15)

,total_product_value,total_freight_value,item_count,freight_pct_of_product_value
order_id,,,,
00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,1,22.563667
00018f77f2f0320c557190d7a144bdd3,239.90,19.93,1,8.307628
000229ec398224ef6ca0657da4fc703e,199.00,17.87,1,8.979899
00024acbcdf0a6daa1e931b038114c75,12.99,12.79,1,98.460354
00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,1,9.074537
00048cc3ae777c65dbb7d2a0634bc1ea,21.90,12.69,1,57.945205
00054e8431b9d7675808bcb819fb4a32,19.90,11.85,1,59.547739
000576fe39319847cbb9d288c5617fa6,810.00,70.75,1,8.734568
0005a1a1728c9d785b8e2b08b904576c,145.95,11.65,1,7.982186


Next, we are trying to investigate as to whether freight value/freight % of product value has an effect on customer satisfaction.

In [171]:
order_value["freight_pct_of_product_value"].describe()

count    98666.000000
mean        30.838869
std         31.476219
min          0.000000
25%         13.186441
50%         22.437396
75%         38.019087
max       2144.705882
Name: freight_pct_of_product_value, dtype: float64

In [173]:
order_value.sort_values(
    "freight_pct_of_product_value",
    ascending=False
).head(10)

,total_product_value,total_freight_value,item_count,freight_pct_of_product_value
order_id,,,,
3ee6513ae7ea23bdfab5b9ab60bffcb5,0.85,18.23,1,2144.705882
6e864b3f0ec71031117ad4cf46b7f2a1,0.85,18.23,1,2144.705882
fb265b2dc558a56445dfc48f8224e201,9.90,121.22,1,1224.444444
d642656598ae928a250620315d19e87e,4.99,37.04,1,742.284569
7c92284adbec8033d160303d83065dbf,9.18,54.69,3,595.751634
b2d30dfaf7114de8ab3f737d062032fe,5.95,34.15,1,573.949580
21fb84496c034392ca2aafe7f7919a3f,6.99,37.90,1,542.203147
a16607f4220f0e60ab4c71b915dd380c,14.00,74.08,2,529.142857
95d6357ffe41aa6d2998852a710c70a0,17.50,91.15,5,520.857143


In [176]:
order_value["freight_pct_of_product_value"].quantile(
    [0.90, 0.95, 0.96, 0.97, 0.98, 0.99, 0.995, 0.999]
)

0.900     62.250306
0.950     83.935520
0.960     90.831687
0.970    101.659553
0.980    117.054264
0.990    146.190858
0.995    180.687831
0.999    281.633929
Name: freight_pct_of_product_value, dtype: float64

In [177]:
reviews_order_value.info()

<class 'pandas.DataFrame'>
Index: 97373 entries, 0 to 99223
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   review_id                97373 non-null  str     
 1   order_id                 97373 non-null  str     
 2   review_score             97373 non-null  int64   
 3   review_comment_title     11445 non-null  str     
 4   review_comment_message   40070 non-null  str     
 5   review_creation_date     97373 non-null  str     
 6   review_answer_timestamp  97373 non-null  str     
 7   total_product_value      97373 non-null  float64 
 8   order_value_band         97373 non-null  category
dtypes: category(1), float64(1), int64(1), str(6)
memory usage: 6.8 MB


In [178]:
reviews_freight = reviews_one_per_order.merge(
    order_value[["total_product_value","total_freight_value","item_count","freight_pct_of_product_value"]],
    on="order_id",
    how="inner"
)

reviews_freight.shape

(97373, 11)

In [179]:
reviews_freight["order_id"].nunique()

97373

In [181]:
reviews_freight["freight_burden_band"] = pd.qcut(
    reviews_freight["freight_pct_of_product_value"],
    q=[0,0.25,0.50,0.75,0.90,1.0],
    labels=["Bottom 25%","25-50%","50-75%","75-90%","Top 10%"]
)

In [184]:
reviews_freight["freight_burden_band"].value_counts().sort_index()

freight_burden_band
Bottom 25%    24350
25-50%        24461
50-75%        24219
75-90%        14607
Top 10%        9736
Name: count, dtype: int64

In [185]:
reviews_freight.head()


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,total_product_value,total_freight_value,item_count,freight_pct_of_product_value,freight_burden_band
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59,370.00,27.26,2,7.367568,Bottom 25%
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13,79.79,8.30,1,10.402306,Bottom 25%
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24,149.00,45.12,1,30.281879,50-75%
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06,179.99,42.85,1,23.806878,50-75%
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53,1199.00,134.25,1,11.196831,Bottom 25%


In [186]:
reviews_freight.groupby("freight_burden_band")["review_score"].agg(["count", "mean"])

,count,mean
freight_burden_band,,
Bottom 25%,24350,4.135688
25-50%,24461,4.118433
50-75%,24219,4.100004
75-90%,14607,4.085233
Top 10%,9736,4.041290


In [187]:
pd.crosstab(
    reviews_freight["freight_burden_band"],
    reviews_freight["review_score"],
    normalize="index"
)*100

review_score,1,2,3,4,5
freight_burden_band,,,,,
Bottom 25%,11.236140,2.854209,7.260780,18.402464,60.246407
25-50%,10.522873,3.131515,8.258043,20.154532,57.933036
50-75%,11.123498,3.121516,8.369462,19.402122,57.983401
75-90%,10.994729,3.327172,8.899843,19.716574,57.061683
Top 10%,11.698850,3.543550,9.326212,19.792523,55.638866


### Finding from the freight % to the customer satisfaction analysis:
- Freight burden shows a weak but consistent association with customer satisfaction.
- Orders in the highest freight-burden group (top 10%) had a lower average review score (4.04) than orders in the lowest 25% (4.14).
- The 5-star review share also declined from 60.25% in the lowest freight-burden group to 55.64% in the top 10%, while the 1-star rate increased only slightly from 11.24% to 11.70%.
- The relationship is relatively small compared with the much stronger relationship observed between delivery performance and review score.
- Freight burden can therefore be considered a secondary factor rather than a primary driver of customer dissatisfaction.

### Next Q: How many customers placed more than one order, and does repeat-customer status appear to be associated with review scores?

In [195]:
orders["order_purchase_timestamp"].dtype

dtype('<M8[us]')

In [199]:
orders_with_customer = orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left"
)

orders_with_customer.head(10)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_time_days,delivery_delay_days,delivery_delay_type,customer_unique_id
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.436574,-8.0,Early,7c396fd4830fd04220f754e42b4e5bff
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.782037,-6.0,Early,af07308b275d755c9edb36a90c618231
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.394213,-18.0,Early,3a653a41f6f9fc3d2a113cf8398680e8
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.208750,-13.0,Early,7c142cf63193a1473d2e66489a9ae977
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.873877,-10.0,Early,72632f0f9dd73dfee390c9b22eb56dd6
5,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01,16.542245,-6.0,Early,80bb27c7c16e8f973207a5086ab329e2
6,136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,NaT,NaT,2017-05-09,NaN,NaN,NaN,36edbb3fb164b1f16485364b6fb04c73
7,6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07,9.989826,-12.0,Early,932afa1e708222e5821dac9cd5db4cae
8,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06,9.818762,-32.0,Early,39382392765b6dc74812866ee5ee92a7
9,e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29 11:55:02,2017-07-29 12:05:32,2017-08-10 19:45:24,2017-08-16 17:14:30,2017-08-23,18.221852,-7.0,Early,299905e3934e9e181bfb2e164dd4b4f8


In [202]:
customer_order_count = orders_with_customer.groupby("customer_unique_id").size()

customer_order_count.head()

customer_unique_id
0000366f3b9a7992bf8c76cfdf3221e2    1
0000b849f77a49e4a4ce2b2a4ca5be3f    1
0000f46a3911fa3c0805444483337064    1
0000f6ccb0745a6a4b88665a16c9f078    1
0004aac84e0df4da2b147fca70cf8255    1
dtype: int64

In [203]:
customer_order_count.describe()


count    96096.000000
mean         1.034809
std          0.214384
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         17.000000
dtype: float64

In [204]:
customer_order_count.value_counts().sort_index()

1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64

In [205]:
orders_with_customer = orders_with_customer.sort_values(by=["customer_unique_id","order_purchase_timestamp"])

In [206]:
orders_with_customer["previous_order_count"] = (
    orders_with_customer
    .groupby("customer_unique_id")
    .cumcount()
)

In [207]:
orders_with_customer[
    [
        "customer_unique_id",
        "order_id",
        "order_purchase_timestamp",
        "previous_order_count"
    ]
].head(20)

,customer_unique_id,order_id,order_purchase_timestamp,previous_order_count
52798,0000366f3b9a7992bf8c76cfdf3221e2,e22acc9c116caa3f2b7121bbb380d08e,2018-05-10 10:56:27,0
73889,0000b849f77a49e4a4ce2b2a4ca5be3f,3594e05a005ac4d06a72673270ef9ec9,2018-05-07 11:11:27,0
26460,0000f46a3911fa3c0805444483337064,b33ec3b699337181488304f362a6b734,2017-03-10 21:05:03,0
98493,0000f6ccb0745a6a4b88665a16c9f078,41272756ecddd9a9ed0180413cc22fb6,2017-10-12 20:29:41,0
41564,0004aac84e0df4da2b147fca70cf8255,d957021f1127559cd947b62533f484f7,2017-11-14 19:45:42,0
78514,0004bd2a26a76fe21f786e4fbd80607f,3e470077b690ea3e3d501cffb5e0c499,2018-04-05 19:33:16,0
74715,00050ab1314c0e55a6ca13cf7181fecf,d0028facea13f508e880202d7097a5a1,2018-04-20 12:57:23,0
6867,00053a61a98854899e70ed204dd4bafe,44e608f2db00c74a1fe329de44416a4e,2018-02-28 11:15:41,0
71235,0005e1862207bf6ccc02e4228effd9a0,ae76bef74b97bcb0b3e355e60d9a6f9c,2017-03-04 23:32:12,0
68876,0005ef4cd20d2893f0d9fbd94d3c0d97,01b330808c5819a6a3cb79b72f0b8288,2018-03-12 15:22:12,0


In [208]:
orders_with_customer["previous_order_count"].value_counts().sort_index()

previous_order_count
0     96096
1      2997
2       252
3        49
4        19
5        11
6         5
7         2
8         2
9         1
10        1
11        1
12        1
13        1
14        1
15        1
16        1
Name: count, dtype: int64

In [210]:
orders_with_customer.shape

(99441, 13)

In [212]:
orders_with_customer["customer_type"] = np.where(
    orders_with_customer["previous_order_count"] == 0,
    "First-time",
    "Repeat"
)

orders_with_customer["customer_type"].value_counts()

customer_type
First-time    96096
Repeat         3345
Name: count, dtype: int64

In [214]:
orders_with_customer["customer_type"].value_counts(normalize="True")*100

customer_type
First-time    96.636196
Repeat         3.363804
Name: proportion, dtype: float64

In [216]:
reviews_customer_type = reviews_one_per_order.merge(
    orders_with_customer[["order_id", "customer_type"]],
    on="order_id",
    how="inner"
)

reviews_customer_type.shape

(98126, 8)

In [217]:
reviews_customer_type["order_id"].nunique()

98126

In [218]:
reviews_customer_type.groupby("customer_type")["review_score"].agg(["count", "mean"])

,count,mean
customer_type,,
First-time,95065,4.084984
Repeat,3061,4.154198


### Conclusion:
Repeat-customer status was investigated as a potential customer-behavior factor. Repeat orders represented only 3.36% of all orders, and their average review score (4.15) was only slightly higher than that of first-time orders (4.08). Given the small difference and limited representation of repeat orders, repeat-customer status does not appear to be a major explanatory factor for review outcomes in this dataset.

### Next question to answer:

Identify orders that are at higher risk of resulting in customer dissatisfaction, using information available about the order and its delivery experience. And since the indicator of customer satisfaction or dissatisfaction is review score here, we are going to classify the review ratings of 1 or 2 as dissatisfied customer. Mixing 3 review scores would mix up clearly negative experiences with somewhat "Neutral" experiences. So, if the reviews are 1 or 2, then it is a dissatisfied customer. And if they are either 3,4 or 5, then it is not counted as a dissatisfied customer.

In [219]:
reviews_one_per_order["dissatisfied"] = np.where(
    reviews_one_per_order["review_score"] <= 2,
    1,
    0
)

reviews_one_per_order["dissatisfied"].value_counts()

dissatisfied
0    83730
1    14396
Name: count, dtype: int64

In [220]:
reviews_one_per_order["dissatisfied"].value_counts(normalize=True) * 100

dissatisfied
0    85.329067
1    14.670933
Name: proportion, dtype: float64

In [235]:
orders["delivery_delay_type"].value_counts()

delivery_delay_type
Early      88649
Late        6535
On time     1292
Name: count, dtype: int64

In [239]:
reviews_delivery = reviews_one_per_order.merge(
    orders[["order_id", "delivery_delay_type"]],
    on="order_id",
    how="inner"
)

reviews_delivery.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,dissatisfied,delivery_delay_type
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59,0,Early
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13,0,Early
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24,0,Early
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06,0,Early
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53,0,Early


In [240]:
reviews_delivery.groupby("delivery_delay_type")["dissatisfied"].mean() * 100

delivery_delay_type
Early       9.208275
Late       62.354422
On time    12.371946
Name: dissatisfied, dtype: float64

62.35% of reviewed late orders received a 1–2 star rating. Hence, it is safe to say that Late delivery is strongly associated with customer dissatisfaction in this dataset.

In [250]:
order_items_with_product_category.columns

Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value',
       'product_category_name'],
      dtype='str')

In [257]:
[name for name in globals() if "order" in name.lower()]

['order_items',
 'orders',
 'items_per_order',
 'multiple_item_orders',
 'multiple_item_orders_pct',
 'payment_types_per_order',
 'orders_per_payment_types',
 'payment_value_per_order',
 'order_value',
 'top_10_extreme_freight_orders',
 'order_items_with_product_category',
 'multiple_review_orders',
 'three_review_orders',
 'multi_review_order_items',
 'multi_review_order_summary',
 'single_review_orders',
 'reviews_one_per_order',
 'reviews_and_orders',
 'late_orders',
 'product_categories_per_orderid',
 'single_category_orders',
 'single_category_order_items',
 'reviews_order_value',
 'missing_order_ids',
 'order_seller',
 'order_seller_count',
 'single_seller_orders',
 'order_seller_single',
 'missing_seller_orders',
 'order_payment_type',
 'customer_order_count',
 'orders_with_customer']

In [276]:
single_category_order_items.info()

single_category_order_items.shape[0]

<class 'pandas.DataFrame'>
Index: 97880 entries, 0 to 112649
Data columns (total 2 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   order_id               97880 non-null  str  
 1   product_category_name  97880 non-null  str  
dtypes: str(2)
memory usage: 2.2 MB


97880

In [285]:
single_category_orders

Index(['00010242fe8c5a6d1ba2dd792cb16214', '00018f77f2f0320c557190d7a144bdd3',
       '000229ec398224ef6ca0657da4fc703e', '00024acbcdf0a6daa1e931b038114c75',
       '00042b26cf59d7ce69dfabb4e55b4fd9', '00048cc3ae777c65dbb7d2a0634bc1ea',
       '00054e8431b9d7675808bcb819fb4a32', '000576fe39319847cbb9d288c5617fa6',
       '0005a1a1728c9d785b8e2b08b904576c', '0005f50442cb953dcd1d21e1fb923495',
       ...
       'fffa82886406ccf10c7b4e35c4ff2788', 'fffb0b1a50e65c449020434fa835e078',
       'fffb2ef8874127f75b52b643880fd7e0', 'fffb9224b6fc7c43ebb0904318b10b5f',
       'fffbee3b5462987e66fb49b1c5411df2', 'fffc94f6ce00a00581880bf54a75a037',
       'fffcd46ef2263f404302a634eb57f7eb', 'fffce4705a9662cd70adb13d4a31832d',
       'fffe18544ffabc95dfada21779c9644f', 'fffe41c64501cc87c801fd61db3f6244'],
      dtype='str', name='order_id', length=97880)

In [283]:
order_category_count = order_items_with_product_category.groupby("order_id")["product_category_name"].nunique()

single_category_orders = order_category_count[order_category_count == 1].index

In [286]:
single_category_order_items = (
    order_items_with_product_category[
        order_items_with_product_category["order_id"].isin(
            single_category_orders
        )
    ][["order_id", "product_category_name"]]
    .drop_duplicates("order_id")
)

In [287]:
reviews_category = reviews_one_per_order.merge(
    single_category_order_items,
    on="order_id",
    how="inner"
)

In [288]:
reviews_category.shape

(96602, 9)

In [289]:
reviews_category.columns

Index(['review_id', 'order_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp', 'dissatisfied', 'product_category_name'],
      dtype='str')

In [290]:
reviews_category["order_id"].nunique()

96602

In [291]:
reviews_category.shape[0]

96602

In [292]:
reviews_category[[
    "dissatisfied",
    "product_category_name"
]].isna().sum()

dissatisfied             0
product_category_name    0
dtype: int64

In [293]:
reviews_category.groupby("product_category_name")["dissatisfied"].mean().sort_values(ascending=False)* 100

product_category_name
seguros_e_servicos                               50.000000
pc_gamer                                         28.571429
portateis_cozinha_e_preparadores_de_alimentos    28.571429
fashion_roupa_masculina                          25.454545
moveis_escritorio                                22.365245
                                                   ...    
livros_interesse_geral                            8.151093
alimentos_bebidas                                 5.882353
musica                                            5.555556
flores                                            3.846154
cds_dvds_musicais                                 0.000000
Name: dissatisfied, Length: 74, dtype: float64

In [294]:
category_dissatisfaction = (
    reviews_category
    .groupby("product_category_name")["dissatisfied"]
    .agg(["count", "mean"])
    .sort_values("mean", ascending=False)
)

category_dissatisfaction["dissatisfaction_pct"] = (
    category_dissatisfaction["mean"] * 100
)

category_dissatisfaction.head(10)

,count,mean,dissatisfaction_pct
product_category_name,,,
seguros_e_servicos,2,0.500000,50.000000
pc_gamer,7,0.285714,28.571429
portateis_cozinha_e_preparadores_de_alimentos,14,0.285714,28.571429
fashion_roupa_masculina,110,0.254545,25.454545
moveis_escritorio,1243,0.223652,22.365245
audio,340,0.211765,21.176471
moveis_colchao_e_estofado,38,0.210526,21.052632
construcao_ferramentas_seguranca,159,0.194969,19.496855
fashion_roupa_feminina,36,0.194444,19.444444


In [298]:
reviews_order_value["dissatisfied"] = np.where(
    reviews_order_value["review_score"] <= 2,
    1,
    0
)

reviews_order_value.groupby("order_value_band")["dissatisfied"].mean() * 100

order_value_band
Low          12.144904
Medium       13.403249
High         13.970649
Very High    17.178344
Name: dissatisfied, dtype: float64

Higher order value is associated with somewhat higher dissatisfaction, with Very High-value orders at 17.18% versus the overall 14.67% baseline. The relationship is noticeable but substantially weaker than the delivery-delay relationship.

In [302]:
seller_reviews.columns

seller_reviews["dissatisfied"] = np.where(
    seller_reviews["review_score"] <= 2,
    1,
    0
)

seller_reviews.groupby("seller_id")["dissatisfied"].mean()*100


seller_id
0015a82c2db000af6aaaf3ae2ecb0532     33.333333
001cca7ae9ae17fb1caed9dfb1094831     15.816327
001e6ad469a905060d959994f1b41e4f    100.000000
002100f778ceb8431b7a1020ff7ab48f     14.583333
003554e2dce176b5555353e4f3555ac8      0.000000
                                       ...    
ffcfefa19b08742c5d315f2791395ee5    100.000000
ffdd9f82b9a447f6f8d4b91554cc7dd3     11.111111
ffeee66ac5d5a62fe688b9d26f83f534     14.285714
fffd5413c0700ac820c7069d66d98c89     20.000000
ffff564a4f9085cd26170f4732393726     65.000000
Name: dissatisfied, Length: 3069, dtype: float64

In [305]:
seller_dissatisfaction = (
    seller_reviews
    .groupby("seller_id")["dissatisfied"]
    .agg(["count", "mean"])
)

seller_dissatisfaction["dissatisfaction_pct"] = (
    seller_dissatisfaction["mean"] * 100
)

seller_dissatisfaction.sort_values("dissatisfaction_pct",ascending=False).head(10)

,count,mean,dissatisfaction_pct
seller_id,,,
ffcfefa19b08742c5d315f2791395ee5,1,1.0,100.0
010da0602d7774602cd1b3f5fb7b709e,1,1.0,100.0
001e6ad469a905060d959994f1b41e4f,1,1.0,100.0
fc99f99385ae90f66f01733856fb88c9,1,1.0,100.0
fc6295add6f51a0936407ead70c1001d,1,1.0,100.0
37ceb404da208418c9032f0674e57804,1,1.0,100.0
391bbd13b6452244774beff1824006ed,2,1.0,100.0
fec6912baad85d41729669edd6b4d3b8,1,1.0,100.0
8501d82f68d23148b6d78bb7c4a42037,1,1.0,100.0


In [307]:
seller_dissatisfaction[seller_dissatisfaction["count"] >= 6].sort_values("dissatisfaction_pct",ascending=False).head(15)

,count,mean,dissatisfaction_pct
seller_id,,,
4342d4b2ba6b161468c63a7e7cfce593,19,0.947368,94.736842
cf8ab1616079e2793aa29d524df01bb1,9,0.777778,77.777778
b1b3948701c5c72445495bd161b83a4c,18,0.777778,77.777778
4b1eaadf791bdbbad8c4a35b65236d52,8,0.750000,75.000000
64b87978a11c1ea7501a89eabe8c2c1a,7,0.714286,71.428571
8bd0e3abda539b9479c4b44a691be1ec,6,0.666667,66.666667
5151aea44289d6c6b090ee31c2132508,6,0.666667,66.666667
6ee85be3693ed79a8e80718743d80655,9,0.666667,66.666667
30c7f28fd3a5897b2c82d152bb760c17,6,0.666667,66.666667


1. Product category showed some variation in customer dissatisfaction. Several categories had dissatisfaction rates above the overall baseline of 14.67%. However, categories with very small numbers of reviewed orders produced unstable percentages, so they were not treated as strong evidence. Higher-volume categories such as moveis_escritorio showed elevated dissatisfaction (22.37%), suggesting that product category may be a secondary risk factor, but its relationship with dissatisfaction was weaker and less consistent than delivery performance.
2. Higher order value was associated with higher customer dissatisfaction. Dissatisfaction increased progressively from 12.14% for Low-value orders to 17.18% for Very High-value orders, compared with an overall baseline of 14.67%. This indicates a moderate relationship, although the difference is substantially smaller than the effect observed for delivery delays.
3. Seller-level dissatisfaction varied substantially, with some sellers showing very high dissatisfaction rates. However, the highest dissatisfaction rates were often based on relatively small numbers of reviewed orders, and dissatisfaction was not concentrated among a small group of sellers. The 10 lowest-rated sellers accounted for only about 0.75% of all one-star reviews. Therefore, seller identity may be useful for identifying individual sellers requiring investigation, but it does not appear to be a strong general-purpose factor for the overall customer dissatisfaction risk score.

In [311]:
reviews_payment["dissatisfied"] = np.where(
    reviews_payment["review_score"] <= 2,
    1,
    0
)

reviews_payment.groupby("payment_type")["dissatisfied"].mean() * 100

payment_type
boleto         14.407257
credit_card    14.692881
debit_card     12.945839
multiple       14.492099
not_defined    66.666667
voucher        18.639798
Name: dissatisfied, dtype: float64

Payment type showed only a weak and inconsistent relationship with customer dissatisfaction. Dissatisfaction rates were relatively close to the overall baseline of 14.67% for most payment types. Voucher payments had a somewhat higher dissatisfaction rate (18.64%), while debit-card payments had a lower rate (12.95%). The not_defined category showed 66.67% dissatisfaction, but this was based on only 3 orders and was therefore not considered meaningful. Overall, payment type was not considered a strong enough factor to include in the customer dissatisfaction risk score.

The overall dissatisfaction rate out of total orders present, which is 14.67%, tells us what the overall population looks like. We compare each factor against that reference to understand whether it is associated with unusually high or low dissatisfaction. Then, using the strength and reliability of the evidence, we decide which factors deserve to enter the risk model and how.

In [314]:
reviews_freight["dissatisfied"] = np.where(
    reviews_freight["review_score"] <= 2,
    1,
    0
)

reviews_freight.groupby("freight_burden_band")["dissatisfied"].mean() * 100

freight_burden_band
Bottom 25%    14.090349
25-50%        13.654389
50-75%        14.245014
75-90%        14.321900
Top 10%       15.242399
Name: dissatisfied, dtype: float64

Freight burden showed little relationship with customer dissatisfaction. Dissatisfaction rates remained close to the overall baseline across all freight-burden groups, ranging from 13.65% to 15.24%. The Top 10% freight-burden group was only slightly above the overall baseline. Therefore, freight burden was not considered a meaningful factor for the customer dissatisfaction risk score.

In [317]:
reviews_customer_type["dissatisfied"] = np.where(
    reviews_customer_type["review_score"] <= 2,
    1,
    0
)

reviews_customer_type.groupby("customer_type")["dissatisfied"].mean() * 100

customer_type
First-time    14.701520
Repeat        13.721006
Name: dissatisfied, dtype: float64

Customer type showed little relationship with dissatisfaction. First-time orders had a dissatisfaction rate of 14.70%, while repeat orders had a slightly lower rate of 13.72%. Both were very close to the overall baseline of 14.67%. Therefore, customer type was not considered a meaningful factor for the customer dissatisfaction risk score.

### The next stage is - Building the risk framework.
We'll answer:
1. Which of the candidate factors actually go into the score?
2. How do we convert their observed behavior into risk levels?
3. How much weight should each factor receive?
4. How do we calculate one final customer/order risk score?
5. Most importantly: does the resulting score actually separate low- and high-dissatisfaction orders when tested against the historical data?

1. Delivery performance -> Very strong association with customer dissatisfaction -> must Include for risk score
2. Order value -> Moderate association	with customer dissatisfaction->️ Candidate for risk score
3. Product category -> Some variation, but inconsistent with customer dissatisfaction -> Candidate for risk score
4. Seller -> Not a strong general predictor of customer dissatisfaction -> Exclude
5. Payment type -> Weak association with customer dissatisfaction -> Exclude
6. Freight burden -> Very weak association with customer dissatisfaction -> Exclude
7. Customer type -> Very weak association with customer dissatisfaction-> Exclude

In [318]:
orders.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'delivery_time_days', 'delivery_delay_days', 'delivery_delay_type'],
      dtype='str')

In [319]:
orders["delivery_delay_days"].describe()

count    96476.000000
mean       -11.876881
std         10.183854
min       -147.000000
25%        -17.000000
50%        -12.000000
75%         -7.000000
max        188.000000
Name: delivery_delay_days, dtype: float64

In [320]:
orders["delivery_delay_type"].value_counts()

delivery_delay_type
Early      88649
Late        6535
On time     1292
Name: count, dtype: int64

In [321]:
orders.loc[orders["delivery_delay_days"] > 0,"delivery_delay_days"].describe()

count    6535.000000
mean       10.620352
std        14.643844
min         1.000000
25%         3.000000
50%         7.000000
75%        13.000000
max       188.000000
Name: delivery_delay_days, dtype: float64

In [323]:
late_orders = orders.loc[
    orders["delivery_delay_days"] > 0
].copy()

late_orders["delay_quartile"] = pd.qcut(
    late_orders["delivery_delay_days"],
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"]
)

late_orders["delay_quartile"].value_counts().sort_index()

delay_quartile
Q1    1870
Q2    1802
Q3    1308
Q4    1555
Name: count, dtype: int64

In [324]:
late_orders_reviewed = late_orders.merge(
    reviews_one_per_order[["order_id", "dissatisfied"]],
    on="order_id",
    how="inner"
)

late_orders_reviewed.groupby("delay_quartile")["dissatisfied"].mean() * 100

delay_quartile
Q1    32.142857
Q2    67.528736
Q3    79.952830
Q4    78.714859
Name: dissatisfied, dtype: float64

In [325]:
late_orders["delivery_delay_days"].quantile([0.25, 0.50, 0.75])

0.25     3.0
0.50     7.0
0.75    13.0
Name: delivery_delay_days, dtype: float64

Delay -> Dissatisfaction
1. 1–3 days	-> 32.14%
2. 4–7 days	-> 67.53%
3. 8–13 days -> 79.95%
4. 14+ days -> 78.71%

We did not arbitrarily decide that "8 days = very high risk." We first observed that dissatisfaction jumps from 32% to 68% around the 4-day boundary, rises further by 8 days, and then largely plateaus. The scoring structure is therefore a simplified representation of the observed relationship.

In [327]:
reviews_order_value["total_product_value"].quantile([0.25, 0.50, 0.75])

0.25     45.9
0.50     86.9
0.75    149.9
Name: total_product_value, dtype: float64

In [329]:
reviews_order_value["value_decile"] = pd.qcut(
    reviews_order_value["total_product_value"],
    q=10,
    labels=False
) + 1

reviews_order_value.groupby("value_decile")["dissatisfied"].mean() * 100

value_decile
1     12.230289
2     12.028470
3     12.590674
4     12.883309
5     14.134999
6     13.500979
7     13.737560
8     15.964135
9     16.170909
10    18.479158
Name: dissatisfied, dtype: float64

Order value has a moderate/secondary relationship with customer dissatisfaction. Higher-value orders tend to have higher dissatisfaction rates, with the highest-value orders showing the clearest increase. However, the relationship is not perfectly monotonic (increasing continuously in simple words) or as strong as the relationship observed for delivery performance.

## Conclusion: Selecting Factors for Deeper Analysis

Exploratory analysis was conducted across delivery performance, order value, product category, seller, payment type, freight burden, and customer type. While some factors showed variation in customer dissatisfaction, the strength, consistency, and reliability of these relationships differed substantially.

Delivery performance showed the strongest and most consistent relationship with dissatisfaction and was therefore selected for deeper analysis.
Order value showed a weaker but persistent upward relationship with dissatisfaction, which was further examined using finer value segments to confirm that the observed pattern was not solely a result of the initial quartile grouping.

For the remaining factors, further segmentation was not pursued because the exploratory results did not provide sufficient evidence of a strong, consistent, or broadly useful relationship that was likely to materially improve the customer-risk framework. In particular, some observed extreme rates were associated with small sample sizes, while other factors showed only modest differences from the overall dissatisfaction baseline.

These factors are therefore excluded from the initial risk-scoring framework rather than subjected to further exploratory segmentation. This does not imply that they have no relationship with customer satisfaction; rather, their observed contribution was not strong enough in this analysis to justify additional complexity in the general-purpose risk score.

In [330]:
delivery_risk = orders[
    ["order_id", "delivery_delay_days"]
].merge(
    reviews_one_per_order[
        ["order_id", "dissatisfied"]
    ],
    on="order_id",
    how="inner"
)

In [335]:
delivery_risk["delivery_risk_level"] = np.select(
    [
        delivery_risk["delivery_delay_days"] <= 0,
        delivery_risk["delivery_delay_days"].between(1, 3),
        delivery_risk["delivery_delay_days"].between(4, 7),
        delivery_risk["delivery_delay_days"] >= 8
    ],
    [
        "Low",
        "Moderate",
        "High",
        "Very High"
    ],
    default="Unknown"

)

delivery_risk["delivery_risk_level"].value_counts()

delivery_risk_level
Low          88951
Unknown       2821
Very High     2766
Moderate      1848
High          1740
Name: count, dtype: int64

The 2,821 Unknown orders are orders where delivery_delay_days is missing (NaN). Remember our earlier investigation: we found 2,821 reviewed orders where order_delivered_customer_date was missing. Since delivery_delay_days was calculated from delivered date − estimated delivery date. if the delivered date is missing, the delay cannot be calculated.

In [336]:
delivery_risk[delivery_risk["delivery_risk_level"] == "Unknown"]["delivery_delay_days"].isna().value_counts()

delivery_delay_days
True    2821
Name: count, dtype: int64

In [339]:
delivery_risk_summary = (
    delivery_risk[
        delivery_risk["delivery_risk_level"] != "Unknown"
    ]
    .groupby("delivery_risk_level")["dissatisfied"]
    .agg(["count", "sum", "mean"])
)

delivery_risk_summary["dissatisfaction_pct"] = delivery_risk_summary["mean"] * 100

delivery_risk_summary

,count,sum,mean,dissatisfaction_pct
delivery_risk_level,,,,
High,1740,1175,0.675287,67.528736
Low,88951,8231,0.092534,9.253409
Moderate,1848,594,0.321429,32.142857
Very High,2766,2193,0.792842,79.284165


In [ ]:
delivery_risk = delivery_risk.merge(
    reviews_order_value[
        ["order_id", "total_product_value"]
    ],
    on="order_id",
    how="left"
)

delivery_risk.head()

In [349]:
delivery_risk.columns.tolist()

['order_id',
 'delivery_delay_days',
 'dissatisfied',
 'delivery_risk_level',
 'total_product_value_x',
 'total_product_value_y',
 'total_product_value']

In [350]:
delivery_risk[
    [
        "total_product_value_x",
        "total_product_value_y",
        "total_product_value"
    ]
].head()

,total_product_value_x,total_product_value_y,total_product_value
0,29.99,29.99,29.99
1,118.70,118.70,118.70
2,159.90,159.90,159.90
3,45.00,45.00,45.00
4,19.90,19.90,19.90


In [351]:
delivery_risk[
    [
        "total_product_value_x",
        "total_product_value_y",
        "total_product_value"
    ]
].isna().sum()

total_product_value_x    753
total_product_value_y    753
total_product_value      753
dtype: int64

In [352]:
delivery_risk = delivery_risk.drop(
    columns=["total_product_value_x", "total_product_value_y"]
)

In [353]:
delivery_risk.columns.tolist()

['order_id',
 'delivery_delay_days',
 'dissatisfied',
 'delivery_risk_level',
 'total_product_value']

In [354]:
delivery_risk = delivery_risk.merge(
    reviews_order_value[
        ["order_id", "value_decile"]
    ],
    on="order_id",
    how="left"
)

In [357]:
delivery_value_analysis = (
    delivery_risk[
        delivery_risk["delivery_risk_level"] != "Unknown"
    ]
    .groupby(
        ["delivery_risk_level", "value_decile"]
    )["dissatisfied"]
    .agg(["count", "mean"])
)

delivery_value_analysis["dissatisfaction_pct"] = (
    delivery_value_analysis["mean"] * 100
)

delivery_value_analysis

count      mean  dissatisfaction_pct
delivery_risk_level value_decile                                      
High                1.0             158  0.708861            70.886076
                    2.0             144  0.666667            66.666667
                    3.0             151  0.668874            66.887417
                    4.0             178  0.674157            67.415730
                    5.0             168  0.660714            66.071429
                    6.0             169  0.639053            63.905325
                    7.0             183  0.650273            65.027322
                    8.0             182  0.659341            65.934066
                    9.0             183  0.726776            72.677596
                    10.0            224  0.691964            69.196429
Low                 1.0            9040  0.079425             7.942478
                    2.0            9055  0.075759             7.575925
                    3.0            8885  0.081261             8.126055
                    4.0            8878  0.079860             7.986033
                    5.0            8927  0.093424             9.342444
                    6.0            8927  0.090736             9.073597
                    7.0            8889  0.086511             8.651142
                    8.0            8771  0.105119            10.511914
                    9.0            8925  0.111036            11.103641
                    10.0           8654  0.123642            12.364225
Moderate            1.0             175  0.285714            28.571429
                    2.0             191  0.256545            25.654450
                    3.0             183  0.333333            33.333333
                    4.0             189  0.359788            35.978836
                    5.0             203  0.330049            33.004926
                    6.0             167  0.365269            36.526946
                    7.0             195  0.343590            34.358974
                    8.0             187  0.294118            29.411765
                    9.0             175  0.377143            37.714286
                    10.0            183  0.273224            27.322404
Very High           1.0             220  0.781818            78.181818
                    2.0             237  0.822785            82.278481
                    3.0             251  0.780876            78.087649
                    4.0             281  0.765125            76.512456
                    5.0             267  0.790262            79.026217
                    6.0             273  0.769231            76.923077
                    7.0             271  0.793358            79.335793
                    8.0             350  0.825714            82.571429
                    9.0             287  0.780488            78.048780
                    10.0            329  0.808511            80.851064

This concludes that delivery delay is the strongest factor we found, and order value doesn't add enough extra information to justify complicating the score. What we tried to do is basically applied the 10 quantiles and respective dissatisfaction % values to each group according to delivery delays. And as it is evident from that, though the dissatisfaction %s vary, there is not a significant upward trend. The data points obviously fluctuate, but that fluctuation is not that significant to be included in the risk score calculation model which is next.

In [358]:
delivery_risk.loc[
    delivery_risk["delivery_risk_level"] == "Unknown",
    ["order_id", "delivery_delay_days", "dissatisfied"]
].head()

,order_id,delivery_delay_days,dissatisfied
6,136cce7faa42fdb2cefd53fdc79a6098,NaN,1
43,ee64d42b8cf066f35eac1cf57de1aa85,NaN,1
102,0760a852e4e9d89eb77bf631eaaf1c84,NaN,0
127,15bed8e2fec7fdbadb186b57c46c92f2,NaN,0
160,36530871a5e80138db53bcfd8a104d90,NaN,1


In [359]:
delivery_risk["delivery_risk_level"].value_counts(dropna=False)

delivery_risk_level
Low          88951
Unknown       2821
Very High     2766
Moderate      1848
High          1740
Name: count, dtype: int64

In [360]:
final_risk = delivery_risk[
    [
        "order_id",
        "delivery_delay_days",
        "delivery_risk_level",
        "dissatisfied"
    ]
].copy()

In [361]:
final_risk["order_id"].is_unique

True

In [362]:
final_risk["dissatisfied"].isna().sum()

np.int64(0)

We are creating a final table that will contain all the relevant data for orders, along with the fact that there is going to be one row per order. And this is the very reason why we are going to exclude the columns such as seller ids, product categories, etc. that are basically having multiple rows per order id. Because our aim is not to recreate a master table that will contain all the information that is there in source tables, but rather to create a table that contains all the important information related to orders that can provide an overview about an order and its risk level related to dissatisfaction.

In [364]:
final_orders = orders[
    [
        "order_id",
        "customer_id",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].copy()

In [365]:
final_orders["delivery_delay_days"] = orders["delivery_delay_days"]

In [366]:
final_orders["delivery_risk_level"] = np.select(
    [
        final_orders["delivery_delay_days"] <= 0,
        final_orders["delivery_delay_days"].between(1, 3),
        final_orders["delivery_delay_days"].between(4, 7),
        final_orders["delivery_delay_days"] >= 8
    ],
    [
        "Low",
        "Moderate",
        "High",
        "Very High"
    ],
    default="Unknown"
)

Here the thing to note is that we directly assigned the column "delivery_delay_days" from orders to final_orders using the operation
final_orders["delivery_delay_days"] = orders["delivery_delay_days"]
But, we recreated the delivery_risk_level using the same method that we used earlier to create it instead of directly assigning it the same way to final_orders. The reason behind this is, we could have done direct assignment if both the tables had similar granularity and indexes and hence same number of records in them in very simple words. Since we copied some columns from orders table into final_orders and our aim is to keep all the orders regardless of whether they have reviews or not, we could safely do it in case of orders. But in case of delivery_risk_level, it was present in delivery_risk dataframe, which had only reviewed orders present in its records and not the ones that had no reviews. Hence, had we tried direct assignment in case of delivery_risk and final_orders, we would get incorrect results, as Pandas assigns data using indexes. And since the indexes would be different for both the tables due to difference in granularity, it is going to create false results. 

In [367]:
final_orders[
    [
        "order_id",
        "delivery_delay_days",
        "delivery_risk_level"
    ]
].head(10)

,order_id,delivery_delay_days,delivery_risk_level
0,e481f51cbdc54678b7cc49136f2d6af7,-8.0,Low
1,53cdb2fc8bc7dce0b6741e2150273451,-6.0,Low
2,47770eb9100c2d0c44946d9cf07ec65d,-18.0,Low
3,949d5b44dbf5de918fe9c16f97b45f8a,-13.0,Low
4,ad21c59c0840e6cb83a9ceb5573f8159,-10.0,Low
5,a4591c265e18cb1dcee52889e2d8acc3,-6.0,Low
6,136cce7faa42fdb2cefd53fdc79a6098,NaN,Unknown
7,6514b8ad8028c9f2cc2374ded245783f,-12.0,Low
8,76c6e866289321a7c93b82b54852dc33,-32.0,Low
9,e69bfb5eb88e0ed6a785585b27e16dbf,-7.0,Low


In [368]:
final_orders["delivery_risk_level"].value_counts(dropna=False)

delivery_risk_level
Low          89941
Unknown       2965
Very High     2863
Moderate      1870
High          1802
Name: count, dtype: int64

In [371]:
order_value.columns

Index(['total_product_value', 'total_freight_value', 'item_count',
       'freight_pct_of_product_value'],
      dtype='str')

In [372]:
order_value.index.name

'order_id'

In [376]:
order_value.drop(columns=["index"])

,order_id,total_product_value,total_freight_value,item_count,freight_pct_of_product_value
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,1,22.563667
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,1,8.307628
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,1,8.979899
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,1,98.460354
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,1,9.074537
...,...,...,...,...,...
98661,fffc94f6ce00a00581880bf54a75a037,299.99,43.41,1,14.470482
98662,fffcd46ef2263f404302a634eb57f7eb,350.00,36.53,1,10.437143
98663,fffce4705a9662cd70adb13d4a31832d,99.90,16.95,1,16.966967
98664,fffe18544ffabc95dfada21779c9644f,55.99,8.72,1,15.574210


In [377]:
order_value["order_id"].is_unique

True

In [378]:
final_orders = final_orders.merge(
    order_value[["order_id","total_product_value","total_freight_value"]],
    on="order_id",
    how="left"
)

In [379]:
final_orders.shape

(99441, 9)

In [380]:
final_orders["order_id"].is_unique

True

In [381]:
order_payment_type.columns

Index(['order_id', 'payment_type_count', 'payment_type'], dtype='str')

In [383]:
order_payment_type["order_id"].is_unique

True

In [384]:
final_orders = final_orders.merge(
    order_payment_type[["order_id","payment_type"]],
    on="order_id",
    how="left"
)

In [388]:
final_orders.shape
final_orders["order_id"].is_unique

True

In [387]:
final_orders.head()

,order_id,customer_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay_days,delivery_risk_level,total_product_value,total_freight_value,payment_type
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,-8.0,Low,29.99,8.72,multiple
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,-6.0,Low,118.70,22.76,boleto
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,-18.0,Low,159.90,19.22,credit_card
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,-13.0,Low,45.00,27.20,credit_card
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,-10.0,Low,19.90,8.72,credit_card


In [389]:
final_orders.isna().sum()

order_id                            0
customer_id                         0
order_purchase_timestamp            0
order_delivered_customer_date    2965
order_estimated_delivery_date       0
delivery_delay_days              2965
delivery_risk_level                 0
total_product_value               775
total_freight_value               775
payment_type                        1
dtype: int64

In [390]:
final_orders[final_orders["total_product_value"].isna()][["order_id", "total_product_value", "total_freight_value"]].head()

,order_id,total_product_value,total_freight_value
266,8e24261a7e58791d10cb1bf9da94df5c,NaN,NaN
586,c272bcd21c287498b4883c7512019702,NaN,NaN
687,37553832a3a89c9b2db59701c357ca67,NaN,NaN
737,d57e15fb07fd180f06ab3926b39edcd2,NaN,NaN
1130,00b1cb0320190ca0daa2c88b35206009,NaN,NaN


In [391]:
final_orders[final_orders["payment_type"].isna()][["order_id", "payment_type"]]

,order_id,payment_type
30710,bfbd0f9bdef84302105ad712db648a6c,NaN


In [393]:
orders_without_items = final_orders[final_orders["total_product_value"].isna()]["order_id"]

orders_without_items.isin(order_items["order_id"]).sum()

np.int64(0)

In [394]:
final_orders.loc[final_orders["payment_type"].isna(),"order_id"]

30710    bfbd0f9bdef84302105ad712db648a6c
Name: order_id, dtype: str

In [400]:
missing_payment_order = final_orders.loc[
    final_orders["payment_type"].isna(),
    "order_id"
]

missing_payment_order.isin(payments["order_id"]).sum()

np.int64(0)

In [401]:
final_orders[
    ["total_product_value", "total_freight_value"]
].describe()

,total_product_value,total_freight_value
count,98666.000000,98666.000000
mean,137.754076,22.823562
std,210.645145,21.650909
min,0.850000,0.000000
25%,45.900000,13.850000
50%,86.900000,17.170000
75%,149.900000,24.040000
max,13440.000000,1794.960000


## Final ETL Dataset

The exploratory analysis identified delivery performance as the strongest factor associated with customer dissatisfaction. Based on this finding, the final ETL output was designed at the **order level**, with one row representing one order.

The final dataset combines core order information with delivery performance, delivery risk classification, order value, freight value, and payment information.

### Final Dataset Grain

**One row = one order**

The final dataset contains 99,441 orders and 10 columns.

### Final Columns

| Column                          | Description |
|---------------------------------|---|
| `order_id`                      | Unique identifier for the order |
| `customer_id`                   | Identifier of the customer associated with the order |
| `order_purchase_timestamp`      | Date and time when the order was placed |
| `order_delivered_customer_date` | Date and time when the order was delivered to the customer |
| `order_estimated_delivery_date` | Estimated delivery date provided for the order |
| `delivery_delay_days`           | Difference between the actual delivery date and estimated delivery date. Negative values indicate early delivery |
| `delivery_risk_level`           | Delivery risk classification derived from delivery delay |
| `total_product_value`           | Total value of products included in the order |
| `total_freight_value`           | Total freight value associated with the order |
| `payment_type`                  | Payment method associated with the order; orders using multiple payment types are classified as `multiple` |

### Delivery Risk Classification

The delivery risk classification was derived from the observed relationship between delivery delay and customer dissatisfaction.

| Delivery delay | Risk level |
|---|---|
| ≤ 0 days | Low |
| 1–3 days | Moderate |
| 4–7 days | High |
| ≥ 8 days | Very High |
| Missing delivery delay | Unknown |

These categories are intended as an **evidence-informed business segmentation**, rather than as statistically validated universal thresholds.

### Handling Missing Values

Missing values were investigated rather than automatically replaced.

- Orders without a delivered customer date cannot have a delivery delay calculated, so their `delivery_delay_days` remains missing and their `delivery_risk_level` is classified as `Unknown`.
- Orders without corresponding `order_items` records have missing `total_product_value` and `total_freight_value`.
- The single order without a corresponding payment record has a missing `payment_type`.
- Missing monetary values were not replaced with zero because missing information does not imply a zero monetary value.

### Data Quality Checks

The final dataset was checked to ensure:

- One row represents one order.
- `order_id` remains unique after all transformations and merges.
- The expected number of orders is preserved.
- Delivery risk categories contain only the defined classifications.
- Missing values were investigated and their causes understood.
- Order value and freight value remain at the order level after merging.

In [406]:
reviews.groupby("order_id")["review_score"].value_counts()

order_id                          review_score
00010242fe8c5a6d1ba2dd792cb16214  5               1
00018f77f2f0320c557190d7a144bdd3  4               1
000229ec398224ef6ca0657da4fc703e  5               1
00024acbcdf0a6daa1e931b038114c75  4               1
00042b26cf59d7ce69dfabb4e55b4fd9  5               1
                                                 ..
fffc94f6ce00a00581880bf54a75a037  5               1
fffcd46ef2263f404302a634eb57f7eb  5               1
fffce4705a9662cd70adb13d4a31832d  5               1
fffe18544ffabc95dfada21779c9644f  5               1
fffe41c64501cc87c801fd61db3f6244  5               1
Name: count, Length: 98875, dtype: int64

In [405]:
reviews.groupby("order_id").size().value_counts().sort_index()


1    98126
2      543
3        4
Name: count, dtype: int64

In [408]:
review_counts = reviews.groupby("order_id").size()

multi_review_orders = review_counts[review_counts > 1].index

reviews[reviews["order_id"].isin(multi_review_orders)][
    ["order_id", "review_id", "review_score"]
].sort_values("order_id")

,order_id,review_id,review_score
25612,0035246a40f520710769010f752e7507,89a02c45c340aeeb1354a24e7d4b2c1e,5
22423,0035246a40f520710769010f752e7507,2a74b0559eb58fc1ff842ecc999594cb,5
22779,013056cfe49763c6f66bda03396c5ee3,ab30810c29da5da8045216f0f62652a2,5
68633,013056cfe49763c6f66bda03396c5ee3,73413b847f63e02bc752b364f6d05ee9,4
854,0176a6846bcb3b0d3aa3116a9a768597,830636803620cdf8b6ffaf1b2f6e92b2,5
...,...,...,...
27465,ff763b73e473d03c321bcd5a053316e8,5e78482ee783451be6026e5cf0c72de1,3
41355,ff850ba359507b996e8b2fbb26df8d03,39de8ad3a1a494fc68cc2d5382f052f4,5
18783,ff850ba359507b996e8b2fbb26df8d03,80f25f32c00540d49d57796fb6658535,5
92230,ffaabba06c9d293a3c614e0515ddbabc,870d856a4873d3a67252b0c51d79b950,3


In [411]:
reviews[reviews["order_id"].isin(multi_review_orders)].sort_values(["order_id","review_creation_date"])

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
22423,2a74b0559eb58fc1ff842ecc999594cb,0035246a40f520710769010f752e7507,5,NaN,Estou acostumada a comprar produtos pelo barat...,2017-08-25 00:00:00,2017-08-29 21:45:57
25612,89a02c45c340aeeb1354a24e7d4b2c1e,0035246a40f520710769010f752e7507,5,NaN,NaN,2017-08-29 00:00:00,2017-08-30 01:59:12
22779,ab30810c29da5da8045216f0f62652a2,013056cfe49763c6f66bda03396c5ee3,5,NaN,NaN,2018-02-22 00:00:00,2018-02-23 12:12:30
68633,73413b847f63e02bc752b364f6d05ee9,013056cfe49763c6f66bda03396c5ee3,4,NaN,NaN,2018-03-04 00:00:00,2018-03-05 17:02:00
854,830636803620cdf8b6ffaf1b2f6e92b2,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30 00:00:00,2018-01-02 10:54:06
...,...,...,...,...,...,...,...
27465,5e78482ee783451be6026e5cf0c72de1,ff763b73e473d03c321bcd5a053316e8,3,NaN,Não sei que haverá acontecido os demais chegaram,2017-11-18 00:00:00,2017-11-18 09:02:48
41355,39de8ad3a1a494fc68cc2d5382f052f4,ff850ba359507b996e8b2fbb26df8d03,5,NaN,Envio rapido... Produto 100%,2017-08-16 00:00:00,2017-08-17 11:56:55
18783,80f25f32c00540d49d57796fb6658535,ff850ba359507b996e8b2fbb26df8d03,5,NaN,"Envio rapido, produto conforme descrito no anu...",2017-08-22 00:00:00,2017-08-25 11:40:22
53962,5476dd0eaee7c4e2725cafb011aa758c,ffaabba06c9d293a3c614e0515ddbabc,3,NaN,NaN,2017-12-20 00:00:00,2017-12-21 13:24:55


In [410]:
reviews.columns.tolist()

['review_id',
 'order_id',
 'review_score',
 'review_comment_title',
 'review_comment_message',
 'review_creation_date',
 'review_answer_timestamp']

In [412]:
multi_review_score_counts = (
    reviews[reviews["order_id"].isin(multi_review_orders)]
    .groupby("order_id")["review_score"]
    .nunique()
)

multi_review_score_counts.value_counts().sort_index()

review_score
1    345
2    202
Name: count, dtype: int64

Q: Why didn't we include review score in our final order-level table?

A: The final dataset was designed at order grain. The review dataset did not always provide one unambiguous score per order—some orders had multiple review records, and some of those had different scores. Rather than arbitrarily averaging or selecting one review, I kept the operational dataset at one row per order and treated customer reviews separately when analysing the relationship between delivery performance and satisfaction.